In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Gold Layer: Enrich, Aggregate & Score (v11 — Bug-Fixed + Enhanced)
# MAGIC
# MAGIC **Input** : `virtue_foundation.ghana.silver_facilities_cleaned`  (112 cols, ~987 rows)
# MAGIC **Outputs**: `virtue_foundation.ghana.gold_facilities_enriched`   (179 cols, survivors only)
# MAGIC             `virtue_foundation.ghana.gold_facilities_audit`      (non-survivor duplicates)
# MAGIC             `virtue_foundation.ghana.gold_regional_summary`      (70 cols)
# MAGIC
# MAGIC ## What changed in v11 (vs v10):
# MAGIC
# MAGIC | #   | Area                        | Fix / Change                                                                      |
# MAGIC |-----|-----------------------------|-----------------------------------------------------------------------------------|
# MAGIC | B1  | CRITICAL                | `transitive_enrich_specialties_udf` was defined but NEVER applied — indentation error inside function body. Now applied correctly after function definition. |
# MAGIC | B2  | CRITICAL                | `import numpy as np` was missing; `compute_all_flags` UDF crashed on `np.ndarray`. |
# MAGIC | B3  | Geo upgrade UDF             | Empty string lat/lon cast to Double produced 0.0 instead of null. Fixed with explicit null-check. |
# MAGIC | B4  | ICU counts                  | has_icu was 1.7% (16 rows) because transitive enrichment never ran. Now properly inferred. |
# MAGIC | F1  | Geo precision tier          | geopy_nominatim upgraded to tier 5 (most precise). text_extracted_city = tier 4. City dicts = tier 3. |
# MAGIC | F2  | Contact completeness        | Now uses `phone_numbers` array length + `official_phone` + `email` triple-check. |
# MAGIC | F3  | Org category quality        | Added `organization_category_inferred` boolean; tightened `private_company` fallback. |
# MAGIC | F4  | Geo region mismatch         | New `geo_region_mismatch` flag when city_clean's known region ≠ region_normalised. |
# MAGIC | F5  | Facility tier label         | New `facility_tier_label` (CHPS/Primary/Secondary/Tertiary/Academic) for RAG queries. |
# MAGIC | F6  | Desert calibration          | Regional desert uses population-weighted penalties; facility-level base credit 0.20. |
# MAGIC | F7  | Ghost probability           | Added `has_name_only` pattern; country_centroid weight increased to 0.20. |
# MAGIC | F8  | Specialty inference         | Added `familyMedicine` fallback for CHPS/clinic type; tighter ICU keyword set. |
# MAGIC | F9  | Address confidence pass-through | Uses silver `address_confidence` column where available.                       |
# MAGIC | F10 | Hospital no-doctors anomaly | Now also requires facility NOT be CHPS or maternity_home (those legitimately have no doctors). |
# MAGIC | F11 | Regional summary            | Added `specialist_hospital_count`, `avg_ghost_probability` per region.            |

# COMMAND ----------
# MAGIC %md ## 0. Imports & Config

# COMMAND ----------

import re
import json
import numpy as np  # FIX B2: was missing, caused NameError in compute_all_flags UDF
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    ArrayType, StringType, FloatType, IntegerType, BooleanType,
    DoubleType, StructType, StructField, LongType,
)
from pyspark.sql.functions import pandas_udf

spark = SparkSession.builder.getOrCreate()
print(f"Run : {datetime.now(timezone.utc).isoformat()}")
print(f"Spark: {spark.version}")

# COMMAND ----------

class Config:
    SILVER          = "virtue_foundation.ghana.silver_facilities_cleaned"
    GOLD_FACILITIES = "virtue_foundation.ghana.gold_facilities_enriched"
    GOLD_AUDIT      = "virtue_foundation.ghana.gold_facilities_audit"
    GOLD_REGIONAL   = "virtue_foundation.ghana.gold_regional_summary"
    EXTRACTION_VER  = "gold_v11"
    MAX_SPECIALTIES_DIRECT_MULTIPLIER = 4   # inferred ≤ direct * 4
    MAX_SPECIALTIES_ABSOLUTE          = 14  # never exceed this regardless

cfg = Config()

CANONICAL_REGIONS = frozenset({
    "Greater Accra", "Ashanti", "Western", "Eastern", "Central",
    "Volta", "Northern", "Upper East", "Upper West",
    "Oti", "Bono East", "Ahafo", "Savannah", "North East",
    "Western North", "Brong-Ahafo",
})
 

# GSS 2021 projected population
GHANA_POPULATION = {
    "Greater Accra":  5_401_964,
    "Ashanti":        5_780_438,
    "Western":        2_376_021,
    "Central":        2_563_228,
    "Eastern":        2_633_154,
    "Volta":          1_693_143,
    "Northern":       2_479_461,
    "Brong-Ahafo":    1_724_847,
    "Upper East":     1_046_545,
    "Upper West":       901_683,
    "Oti":              663_338,
    "Bono East":      1_141_427,
    "Ahafo":            564_397,
    "Savannah":         606_986,
    "North East":       527_170,
    "Western North":    782_022,
    "Unknown":        1_000_000,  # kept for audit queries only
}

EVIDENCE_WEIGHT_MAP = {
    "structured":      1.00,
    "text_extracted":  0.70,
    "inferred":        0.50,
    "social_metadata": 0.30,
}

# ── City→region lookup for geo_region_mismatch detection (F4) ──────────────
# A representative sample; extend as needed
_CITY_TO_REGION = {
    # Greater Accra
    "accra":"Greater Accra","tema":"Greater Accra","ashaiman":"Greater Accra","madina":"Greater Accra",
    "east legon":"Greater Accra","north legon":"Greater Accra","cantonments":"Greater Accra","dansoman":"Greater Accra",
    "achimota":"Greater Accra","lapaz":"Greater Accra","spintex":"Greater Accra","osu":"Greater Accra",
    "adenta":"Greater Accra","teshie":"Greater Accra","nungua":"Greater Accra","adabraka":"Greater Accra",
    "jamestown":"Greater Accra","labadi":"Greater Accra","kokomlemle":"Greater Accra","amasaman":"Greater Accra",
    "kwabenya":"Greater Accra","dome":"Greater Accra","oyarifa":"Greater Accra","airport residential":"Greater Accra",
    "awoshie":"Greater Accra","weija":"Greater Accra","haatso":"Greater Accra","east cantonments":"Greater Accra",
    "roman ridge":"Greater Accra","kaneshie":"Greater Accra","north kaneshie":"Greater Accra",
    "darkuman":"Greater Accra","chorkor":"Greater Accra","okponglo":"Greater Accra","legon":"Greater Accra",
    "agbogba":"Greater Accra","mempeasem":"Greater Accra","ashale-botwe":"Greater Accra","dzorwulu":"Greater Accra",
    "klagon":"Greater Accra","odorkor":"Greater Accra","pokoase":"Greater Accra","pantang":"Greater Accra",
    "accra central":"Greater Accra","agbogbloshie":"Greater Accra","tesano":"Greater Accra","labone":"Greater Accra",
    "ridge":"Greater Accra","airport city":"Greater Accra","accra newtown":"Greater Accra","dodowa":"Greater Accra",
    "kpone":"Greater Accra","la":"Greater Accra","pokuase":"Greater Accra","ga south":"Greater Accra",
    "ga east":"Greater Accra","kwashieman":"Greater Accra","manhean":"Greater Accra","adentan":"Greater Accra",
    "ashiaman":"Greater Accra","sowutuom":"Greater Accra","abokobi":"Greater Accra","ablekuma":"Greater Accra",
    "kasoa":"Central",  
    "kumasi":"Ashanti","obuasi":"Ashanti","ejisu":"Ashanti","asokore":"Ashanti","atonsu":"Ashanti",
    "suame":"Ashanti","bantama":"Ashanti","nhyiaeso":"Ashanti","asawase":"Ashanti","tafo":"Ashanti",
    "kwadaso":"Ashanti","mampong":"Ashanti","nkawie":"Ashanti","kaase":"Ashanti","bekwai":"Ashanti",
    "agona ashanti":"Ashanti","abuakwa":"Ashanti","agogo":"Ashanti","santasi":"Ashanti","buokrom":"Ashanti",
    "manhyia":"Ashanti","asokwa":"Ashanti","kronum":"Ashanti","ahodwo":"Ashanti","offinso":"Ashanti",
    "kumawu":"Ashanti","konongo":"Ashanti","tepa":"Ashanti","juaben":"Ashanti",
    "kenyase":"Ashanti","asokore mampong":"Ashanti","ejisu juaben":"Ashanti",
    # v13 new Ashanti entries
    "asuofia":"Ashanti","asuofua":"Ashanti","kuntanase":"Ashanti","sekyere":"Ashanti",
    "juaso":"Ashanti","ejura":"Ashanti","tikrom":"Ashanti","mankranso":"Ashanti",
    # Western
    "takoradi":"Western","sekondi":"Western","axim":"Western","tarkwa":"Western","half assini":"Western",
    "prestea":"Western","bogoso":"Western","sefwi asawinso":"Western","sefwi wiawso":"Western",
    "daboase":"Western","apremdo":"Western","aboadze":"Western","kwesimintsim":"Western",
    "mataheko":"Western","new takoradi":"Western","nsuta":"Western","sekondi-takoradi":"Western",
    "shama":"Western","agona nkwanta":"Western","inchaban":"Western","apowa":"Western",
    "benso":"Western","dompim":"Western","nsuta-wassaw":"Western","dompoase":"Western",
    # Western North
    "bibiani":"Western North","sefwi bodi":"Western North","juaboso":"Western North",
    "anhwiaso":"Western North","enchi":"Western North","sefwi bekwai":"Western North",
    "asankrangua":"Western North","sefwi boinzan":"Western North","boinzan":"Western North",
    # Eastern
    "koforidua":"Eastern","nkawkaw":"Eastern","suhum":"Eastern","somanya":"Eastern",
    "akyem oda":"Eastern","kade":"Eastern","asamankese":"Eastern","mpraeso":"Eastern",
    "abetifi":"Eastern","akosombo":"Eastern","mampong-akwapim":"Eastern","nsawam":"Eastern",
    "nsawam adoagyiri":"Eastern","akim oda":"Eastern","oda":"Eastern","nkwantanan":"Eastern",
    "kyebi":"Eastern","begoro":"Eastern","akim swedru":"Eastern","akwatia":"Eastern",
    # v13 new Eastern entries
    "abomosu":"Eastern","kwabeng":"Eastern","new abirim":"Eastern","enyinabrim":"Eastern",
    # Central
    "cape coast":"Central","winneba":"Central","saltpond":"Central","ajumako":"Central",
    "mankessim":"Central","ankaful":"Central","buduburam":"Central","elmina":"Central",
    "agona swedru":"Central","assin fosu":"Central","cabo corso":"Central","cape-coast":"Central",
    "awutu bereku":"Central","bawjiase":"Central","dunkwa":"Central","assin north":"Central",
    "twifo ati-morkwa":"Central","swedru":"Central","breman asikuma":"Central",
    # v13 new Central entries
    "abura":"Central","asin":"Central","abura asebu":"Central","assin":"Central",
    "nyakrom":"Central","gomoa":"Central","mumford":"Central","apam":"Central",
    # Volta
    "ho":"Volta","hohoe":"Volta","keta":"Volta","akatsi":"Volta","aflao":"Volta","kpandu":"Volta",
    "sogakope":"Volta","battor":"Volta","anloga":"Volta","adidome":"Volta","anfoega":"Volta",
    "jasikan":"Volta","denu":"Volta","dzodze":"Volta","ve golokwati":"Volta",
    "kpeta":"Volta","peki":"Volta","nkonya":"Volta","ateiku":"Volta",
    # Oti
    "dambai":"Oti","nkwanta":"Oti","yabologu":"Oti","kpassa":"Oti","buem":"Oti",
    # Northern
    "tamale":"Northern","walewale":"Northern","yendi":"Northern","savelugu":"Northern",
    "tolon":"Northern","saboba":"Northern","bimbila":"Northern","gushegu":"Northern",
    "karaga":"Northern","kumbungu":"Northern","kpandai":"Northern",
    # Savannah
    "salaga":"Savannah","damongo":"Savannah","bole":"Savannah","sawla":"Savannah","buipe":"Savannah",
    # North East
    "nalerigu":"North East","kparigu":"North East","chereponi":"North East",
    # Bono East
    "techiman":"Bono East","nkoranza":"Bono East","kintampo":"Bono East","atebubu":"Bono East",
    "yeji":"Bono East","prang":"Bono East","anyinasusu":"Bono East","anyinasuso":"Bono East",
    # Bono / Brong-Ahafo
    "sunyani":"Bono","berekum":"Bono","banda":"Bono","wenchi":"Bono","dormaa ahenkro":"Bono",
    "abesim":"Bono","dormaa":"Bono","jaman north":"Bono","jaman south":"Bono",
    # Ahafo
    "goaso":"Ahafo","bechem":"Ahafo","duayaw nkwanta":"Ahafo","kukuom":"Ahafo",
    "kenyasi":"Ahafo","acherensua":"Ahafo","mim":"Ahafo","hwidiem":"Ahafo","mepom":"Ahafo",
    # Upper East
    "bolgatanga":"Upper East","navrongo":"Upper East","bawku":"Upper East","zebilla":"Upper East",
    "sandema":"Upper East","bongo":"Upper East","paga":"Upper East","chiana":"Upper East",
    "tongo":"Upper East",
    # Upper West
    "wa":"Upper West","lawra":"Upper West","nandom":"Upper West","jirapa":"Upper West",
    "nadawli":"Upper West","daffiama":"Upper West","wechiau":"Upper West",
    "kaleo":"Upper West","issa":"Upper West","gwolu":"Upper West","eremon":"Upper West",
}

# ── Extended city coordinates (v10 + additions) ────────────────────────────
CITY_COORDS_EXTENDED = {
    # Greater Accra suburbs/areas
    "Achimota":           (5.6300, -0.2333), "Adentan":            (5.7000, -0.1833),
    "Adentan-Fahraha":    (5.7100, -0.1800), "Adenta-Fafraha":     (5.7100, -0.1800),
    "Agbogba":            (5.7167, -0.2000), "Agbogbloshie":       (5.5500, -0.2167),
    "Ahodwo":             (6.6500, -1.6333), "Airport Residential":(5.6000, -0.1667),
    "Amasaman":           (5.7000, -0.2667), "Cantonments":        (5.5833, -0.1833),
    "Community":          (5.6500, -0.0167), "Darkuman-Nyamekye":  (5.5833, -0.2333),
    "Dzorwulu":           (5.6000, -0.2000), "East Airport":       (5.6167, -0.1667),
    "Haatso":             (5.6833, -0.2000), "James Town":         (5.5500, -0.2167),
    "Klagon":             (5.6167, -0.0333), "Kwadaso":            (6.6667, -1.6667),
    "Labadi":             (5.5667, -0.1333), "Mempeasem":          (5.7167, -0.2167),
    "Nima":               (5.5833, -0.2000), "North Kaneshie":     (5.5833, -0.2500),
    "North Legon":        (5.6667, -0.1667), "Accra Newtown":      (5.5667, -0.2000),
    "Ofankor":            (5.6667, -0.2667), "Oyarifa":            (5.7333, -0.1667),
    "Pantang":            (5.7333, -0.0833), "Ridge":              (5.5667, -0.1833),
    "Abelenkpe, Accra":   (5.6167, -0.1833), "Adenta":             (5.7000, -0.1833),
    "Buokrom":            (6.7000, -1.6000), "Kronum":             (6.7000, -1.6500),
    "Asokore":            (6.7167, -1.5833), "Asokore Mampong":    (6.7500, -1.5500),
    "Bantama":            (6.7000, -1.6333), "Manhyia":            (6.7167, -1.6167),
    "Suame":              (6.7167, -1.6500), "Tafo":               (6.7333, -1.6000),
    "Nkawie":             (6.6000, -1.8500), "Bekwai":             (6.4500, -1.5833),
    "Offinso":            (7.1167, -1.6333), "Konongo":            (6.6167, -1.2167),
    "Juaben":             (6.8000, -1.4167), "Mampong-Akwapim":    (5.9167, -0.1333),
    "Akosombo":           (6.2833,  0.0500), "Akwatia":            (6.0333, -0.8000),
    "Akwatialine":        (6.0333, -0.8167), "Asamang":            (6.8000, -1.1667),
    "Tepa":               (7.0000, -2.5000), "Kuntanase":          (6.6167, -1.5333),
    "Mankranso":          (7.0500, -2.0500), "Tikrom":             (6.5833, -1.3833),
    "Sekyere":            (7.1000, -1.3000), "Juaso":              (6.5833, -1.0833),
    "Ejura":              (7.3833, -1.3667), "Kwabeng":            (6.4000, -0.7833),
    "New Abirim":         (6.4333, -0.8167),
    # Western Region
    "Sefwi Asawinso":     (6.3500, -2.4000), "Sefwi Bekwai":       (6.3833, -2.3833),
    "Sefwi Wiawso":       (6.2000, -2.4833), "Sefwi Bodi":         (6.0833, -2.5167),
    "Apowa":              (4.8500, -1.9833), "Apremdo":            (4.8333, -1.9500),
    "Benso":              (5.1833, -2.1500), "Dompim":             (5.1667, -2.0167),
    "Dompoase":           (5.2000, -2.1000), "Mataheko":           (4.9167, -1.7500),
    "New Takoradi":       (4.9000, -1.7667), "Dixcove":            (4.8167, -1.9667),
    "Asankrangua":        (5.6833, -2.5167), "Daboase":            (5.0667, -1.7000),
    "Nsuta":              (5.2167, -1.8167),
    # Eastern Region
    "Adoagyiri-Adeiso":   (5.8000, -0.5000), "Achiase":            (6.2667, -0.5500),
    "Aframso":            (7.1833, -1.6167), "Enyinabrim":         (6.8500, -1.2333),
    "Nsawura":            (6.5000, -0.8833), "Breman Asikuma":     (5.7333, -1.1167),
    "Nkenkaso":           (6.2333, -0.8667), "Twabidi":            (6.5667, -1.4333),
    "Kokofu":             (6.8833, -1.7833), "Mpatuom":            (6.1167, -0.8333),
    # Volta Region
    "Adidome":            (5.8833,  0.5833), "Anloga":             (5.7333,  0.8833),
    "Anfoega":            (6.9500,  0.3667), "Battor":             (6.2000,  0.5000),
    "Peki":               (6.9333,  0.3667), "Nkonya":             (7.1500,  0.4167),
    "Kpando":             (6.9833,  0.3000), "Ateiku":             (6.8000,  0.3333),
    # Central Region
    "Ajumako":            (5.3167, -1.0500), "Dunkwa-On-Offin":    (5.9667, -1.7833),
    "Adum Banso":         (5.2167, -1.5333),
    # Northern Region
    "Karaga":             (10.0167, -0.5333), "Kpandai":            (8.4667, -0.0167),
    "Kabiase Gonja":      (9.4000, -1.0000), "Kamgbunli":          (9.8000, -1.1667),
    # Savannah / NE
    "Bimbilla":           (9.8667,  0.0667), "Damongo":            (9.0833, -1.8167),
    "Bole":               (9.0167, -2.4833),
    # Upper West
    "Nadawli":            (10.6000, -2.7500), "Daffiama":           (10.4500, -2.4500),
    # Bono / Brong-Ahafo
    "Abesim":             (7.3000, -2.3167), "Dormaa":             (7.3000, -2.7167),
    "Banda":              (7.7833, -2.2833), "Anyinasuso":         (7.5000, -1.7500),
    "Anyinasusu":         (7.5000, -1.7500), "Ahimakrom":          (7.1500, -2.2500),
    # Ahafo
    "Acherensua":         (6.8167, -2.4833), "Ayanfuri":           (6.0833, -1.6833),
    "Mepom":              (6.9500, -2.2833), "Adumkrom":           (6.9000, -2.3000),
    # Oti Region
    "Yabologu":           (8.1833,  0.4167),
    # Misc
    "Darkuman":           (5.6000, -0.2333), "Tema Community 22":  (5.6700, -0.0166),
    "Juaboso":            (6.4833, -2.5833), "Bakanta":            (5.3000, -1.1500),
    "Akaporiso":          (9.8500,  0.0000), "Mamudukrom":         (6.4833, -0.9500),
    "Odonkawkrom":        (6.4167, -1.6500), "Nyinamponase":       (6.6833, -1.5500),
    "Brebre":             (6.9833, -2.0333),
}

_SILVER_CITY_COORDS = {
    "Accra": (5.6037,-0.1870), "Tema": (5.6698,-0.0166), "Ashaiman": (5.7000,-0.0333),
    "Madina": (5.6833,-0.1667), "East Legon": (5.6333,-0.1667), "Dome": (5.6667,-0.2333),
    "Dansoman": (5.5667,-0.2500), "Nungua": (5.5833,-0.0667), "Teshie": (5.5833,-0.0833),
    "Osu": (5.5667,-0.1833), "Kumasi": (6.6885,-1.6244), "Obuasi": (6.2000,-1.6667),
    "Ejisu": (6.7167,-1.5833), "Mampong": (7.0667,-1.4000), "Takoradi": (4.8845,-1.7554),
    "Sekondi": (4.9333,-1.7000), "Axim": (4.8678,-2.2378), "Tarkwa": (5.3003,-1.9936),
    "Cape Coast": (5.1053,-1.2466), "Winneba": (5.3500,-0.6333), "Saltpond": (5.2000,-1.0667),
    "Mankessim": (5.2667,-1.0167), "Kasoa": (5.5100,-0.4350), "Ho": (6.6000, 0.4667),
    "Hohoe": (7.1500, 0.4667), "Keta": (5.9167, 0.9833), "Akatsi": (6.1167, 0.8000),
    "Sogakope": (5.8833, 0.5833), "Aflao": (6.1167, 1.1833), "Tamale": (9.4075,-0.8533),
    "Yendi": (9.4417,-0.0083), "Walewale": (10.3667,-0.8000), "Savelugu": (9.6167,-0.8167),
    "Bolgatanga": (10.7861,-0.8522), "Navrongo": (10.9000,-1.0833), "Bawku": (11.0583,-0.2417),
    "Wa": (10.0600,-2.5000), "Lawra": (10.6333,-2.9000), "Jirapa": (10.5333,-2.7500),
    "Nandom": (10.8500,-2.7500), "Koforidua": (6.0886,-0.2594), "Nkawkaw": (6.5500,-0.7667),
    "Suhum": (6.0333,-0.4500), "Somanya": (6.1000,-0.0167), "Sunyani": (7.3333,-2.3167),
    "Berekum": (7.4500,-2.5833), "Techiman": (7.5833,-1.9333), "Nkoranza": (7.6667,-1.7000),
    "Kintampo": (8.0500,-1.7167), "Atebubu": (7.7500,-0.9833), "Goaso": (6.8000,-2.5167),
    "Bechem": (7.1500,-2.3000), "Nalerigu": (10.5167,-0.3500), "Damongo": (9.0833,-1.8167),
    "Bole": (9.0167,-2.4833), "Dambai": (8.0667, 0.1833), "Nsawam": (5.8000,-0.3500),
    "Dodowa": (5.9000,-0.1167), "Weija": (5.5833,-0.3000), "Tesano": (5.6167,-0.2167),
    "Lapaz": (5.6167,-0.2500), "Agogo": (6.8000,-1.1000), "Kpone": (5.6667,-0.0167),
    "Bibiani": (6.4500,-2.3500), "Wenchi": (7.7500,-2.1000), "Dormaa Ahenkro": (7.3000,-2.7167),
    "Atonsu": (6.6500,-1.6333), "Adenta": (5.7000,-0.1833), "Santasi": (6.6667,-1.6167),
    "Kwashieman": (5.6000,-0.2333), "Abuakwa": (6.0167,-0.2167), "Pokuase": (5.6833,-0.2667),
}

_ALL_CITY_COORDS = {**_SILVER_CITY_COORDS, **CITY_COORDS_EXTENDED}

# Build lookup dict (lowercase keys)
_city_coord_map = {k.lower().strip(): v for k, v in _ALL_CITY_COORDS.items()}

# COMMAND ----------
# MAGIC %md ## 1. Read Silver + Contract Validation

# COMMAND ----------

silver = spark.table(cfg.SILVER)
_raw_count = silver.count()
_col_count  = len(silver.columns)
print(f"Silver rows    : {_raw_count:,}")
print(f"Silver columns : {_col_count}")

REQUIRED_SILVER_COLUMNS = [
    "unique_id", "name",
    "facility_type_clean", "organization_category",
    "is_ngo", "is_faith_based", "is_government",
    "operatorTypeId", "operational_status", "is_operational",
    "address_type", "has_physical_address", "postal_address",
    "row_quality_flags", "source_trust",
    "dedup_key", "is_duplicate_survivor",
    "specialties_parsed", "procedure_parsed", "equipment_parsed", "capability_parsed",
    "region_normalised", "latitude", "longitude", "geo_source", "geo_confidence",
    "data_completeness_score", "doc_text_length", "is_rag_ready",
    "citations", "extraction_version",
    "number_doctors_int", "capacity_int",
    "facility_type_raw", "operator_type_raw", "facility_type_clean_pdf",
    "dedup_cluster_size", "is_generated_canonical", "canonical_source_group",
    "year_established_int", "year_confidence",
]

_missing = [c for c in REQUIRED_SILVER_COLUMNS if c not in silver.columns]
if _missing:
    raise ValueError(f"Silver contract violation — {len(_missing)} missing:\n" + "\n".join(f"  {c}" for c in _missing))
print("✅  Silver contract passed.")

# COMMAND ----------
# MAGIC %md ## 2. Write Audit Table + Filter to Survivors

# COMMAND ----------

dup_audit = silver.filter(~F.col("is_duplicate_survivor"))
_audit_count = dup_audit.count()
print(f"Non-survivor rows (→ audit): {_audit_count:,}")
 
(dup_audit.write.format("delta").mode("overwrite")
 .option("overwriteSchema", "true").saveAsTable(cfg.GOLD_AUDIT))
 
silver = silver.filter(F.col("is_duplicate_survivor"))
_survivor_count = silver.count()
print(f"Survivor rows: {_survivor_count:,}")

# COMMAND ----------
# MAGIC %md ## 3. Second-Pass Region Consolidation

# COMMAND ----------

_REGION_SECOND_PASS = {
    "Bono": "Brong-Ahafo", "Bono Region": "Brong-Ahafo",
    "Ga East Municipality": "Greater Accra",
    "Ga East Municipality, Greater Accra": "Greater Accra",
    "Shai Osudoku District, Greater Accra": "Greater Accra",
    "Ledzokuku-Krowor": "Greater Accra", "Tema West Municipal": "Greater Accra",
    "Accra East": "Greater Accra", "Accra North": "Greater Accra",
    "East Legon": "Greater Accra", "North Legon": "Greater Accra",
    "Adenta Municipal": "Greater Accra",
    "Asokore Mampong Municipal": "Ashanti", "Ejisu Municipal": "Ashanti",
    "Ejisu-Juaben Municipal": "Ashanti", "Ahafo Ano South-East": "Ashanti",
    "Ahafo Ano South East": "Ashanti", "Kumasi Metropolitan": "Ashanti",
    "ASHANTI": "Ashanti", "SH": "Ashanti",
    "KEEA": "Central", "Cape Coast Municipal": "Central", "Central Ghana": "Central",
    "Dormaa East": "Brong-Ahafo", "Techiman Municipal": "Bono East",
    "Sissala West District": "Upper West", "Sissala East District": "Upper West",
    "Central Tongu District": "Volta", "South Tongu": "Volta",
    "Tamale Metropolitan": "Northern",
}

_rmap_expr = F.create_map(
    *[F.lit(x) for pair in _REGION_SECOND_PASS.items() for x in pair]
)

silver = silver.withColumn(
    "region_normalised",
    F.when(
        F.col("region_normalised").isNull() | (F.trim(F.col("region_normalised")) == ""),
        F.lit("Unknown"),
    ).when(
        F.col("region_normalised").isin(list(CANONICAL_REGIONS)),
        F.col("region_normalised"),
    ).when(
        _rmap_expr[F.col("region_normalised")].isNotNull(),
        _rmap_expr[F.col("region_normalised")],
    ).otherwise(F.lit("Unknown"))
)

# COMMAND ----------
# MAGIC %md ## 4. Geo Upgrade — Extended City Dictionary
# MAGIC
# MAGIC FIX B3: properly null-out lat/lon when source string is empty/invalid.

# COMMAND ----------

@pandas_udf(ArrayType(StringType()))
def upgrade_geo_from_city_udf(
    city_col: pd.Series, geo_src_col: pd.Series,
    lat_col: pd.Series, lon_col: pd.Series
) -> pd.Series:
    """
    Only upgrades rows where geo_source is region_centroid or country_centroid.
    Returns [lat_str, lon_str, geo_source_str, geo_confidence_str].
    FIX B3: empty-string lat/lon now returns 'NULL' sentinel instead of '0.0'.
    """
    result = []
    for city_v, geo_src, lat_v, lon_v in zip(city_col, geo_src_col, lat_col, lon_col):
        geo_src_str = str(geo_src or "").strip()

        # FIX B3: safe float conversion; return None sentinel for bad values
        def _safe_float(x):
            try:
                val = float(x)
                return None if (val == 0.0 or str(x).strip() == "") else val
            except (TypeError, ValueError):
                return None

        lat_safe = _safe_float(lat_v)
        lon_safe = _safe_float(lon_v)

        # Only upgrade if currently region or country centroid
        if geo_src_str not in ("region_centroid", "country_centroid"):
            result.append([
                str(lat_safe) if lat_safe is not None else "",
                str(lon_safe) if lon_safe is not None else "",
                geo_src_str,
                "0.90",
            ])
            continue

        city_lower = str(city_v or "").strip().lower()
        if city_lower and city_lower in _city_coord_map:
            lat, lon = _city_coord_map[city_lower]
            result.append([str(lat), str(lon), "extended_city_dict", "0.85"])
        else:
            confidence = "0.40" if "region" in geo_src_str else "0.15"
            result.append([
                str(lat_safe) if lat_safe is not None else "",
                str(lon_safe) if lon_safe is not None else "",
                geo_src_str,
                confidence,
            ])
    return pd.Series(result)


_geo_upg = upgrade_geo_from_city_udf(
    F.col("city_clean"), F.col("geo_source"), F.col("latitude"), F.col("longitude")
)
silver = silver \
    .withColumn("_gu", _geo_upg) \
    .withColumn("latitude",
        F.when(F.col("_gu").getItem(0) != "", F.col("_gu").getItem(0).cast(DoubleType()))
         .otherwise(F.lit(None).cast(DoubleType()))
    ) \
    .withColumn("longitude",
        F.when(F.col("_gu").getItem(1) != "", F.col("_gu").getItem(1).cast(DoubleType()))
         .otherwise(F.lit(None).cast(DoubleType()))
    ) \
    .withColumn("geo_source",   F.col("_gu").getItem(2)) \
    .withColumn("geo_confidence", F.col("_gu").getItem(3).cast(DoubleType())) \
    .drop("_gu")

# ── v11-F4: geo_region_mismatch flag ──────────────────────────────────────
_city_region_map_expr = F.create_map(
    *[F.lit(x) for pair in _CITY_TO_REGION.items() for x in pair]
)

silver = silver.withColumn(
    "geo_region_mismatch",
    F.when(
        F.col("region_normalised").isin(list(CANONICAL_REGIONS))
        & F.col("city_clean").isNotNull()
        & (_city_region_map_expr[F.lower(F.trim(F.col("city_clean")))].isNotNull()),
        F.when(
            _city_region_map_expr[F.lower(F.trim(F.col("city_clean")))] != F.col("region_normalised"),
            F.lit(True)
        ).otherwise(F.lit(False))
    ).otherwise(F.lit(False))
).cast("boolean") if False else silver.withColumn(
    "geo_region_mismatch",
    F.when(
        F.col("region_normalised").isin(list(CANONICAL_REGIONS))
        & F.col("city_clean").isNotNull()
        & (_city_region_map_expr[F.lower(F.trim(F.col("city_clean")))].isNotNull())
        & (_city_region_map_expr[F.lower(F.trim(F.col("city_clean")))] != F.col("region_normalised")),
        F.lit(True)
    ).otherwise(F.lit(False))
)

print("Region distribution after second-pass:")
silver.groupBy("region_normalised").count().orderBy(F.desc("count")).show(20)
_unk_ct = silver.filter(F.col("region_normalised") == "Unknown").count()
print(f"Unknown: {_unk_ct}/{_survivor_count} ({_unk_ct/_survivor_count*100:.1f}%)  target <15%")

print("\nGeo source after city-dict upgrade:")
silver.groupBy("geo_source").count().orderBy(F.desc("count")).show()

_mismatch_ct = silver.filter(F.col("geo_region_mismatch")).count()
print(f"geo_region_mismatch: {_mismatch_ct} rows flagged")

# COMMAND ----------
# MAGIC %md ## 5. Fix Nullable Boolean Columns

# COMMAND ----------

for _bool_col, _cat_val in [
    ("is_ngo",         "ngo"),
    ("is_faith_based", "faith_based"),
    ("is_government",  "government"),
]:
    silver = silver.withColumn(
        _bool_col,
        F.coalesce(
            F.col(_bool_col),
            F.when(F.lower(F.trim(F.col("organization_category"))) == _cat_val, F.lit(True)),
            F.lit(False),
        )
    )

silver = silver.withColumn(
    "is_operational", F.coalesce(F.col("is_operational"), F.lit(True))
)

# COMMAND ----------
# MAGIC %md ## 6. Organization Category Expansion (v11)
# MAGIC
# MAGIC v11 adds `organization_category_inferred` boolean to distinguish original vs inferred.
# MAGIC Tightened `private_company` fallback — only apply when operatorTypeId is not "public".

# COMMAND ----------

_GOVT_RE = re.compile(
    r"""(?xi)
    \b(
      ghana\s+health\s+service | GHS\b | government | polyclinic |
      national\s+(?:hospital|health|health\s+service|blood\s+service|ambulance|insurance|laboratory) |
      (?:municipal|district|regional|sub.?district)\s+(?:hospital|health|clinic|medical\s+cent) |
      (?:district|municipal|regional)\s+(?:medical|health) |
      police\s+(?:hospital|clinic|service|medical) |
      prison\s+(?:service|hospital) |
      military\s+(?:hospital|clinic|medical) |
      armed\s+forces\s+(?:hospital|medical) |
      [0-9]+BN\s+(?:Military\s+)?Hospital |
      37\s+(?:Military\s+)?Hospital |
      community\s+health\s+(?:compound|planning|service|center|post) |
      CHPS\s+compound | CHPS\b |
      ministry\s+of\s+health | MOH\b |
      public\s+(?:hospital|health\s+facility|clinic) |
      health\s+center\s+(?:no|number|\d+) |
      teaching\s+hospital | referral\s+hospital |
      ridge\s+hospital | korle\s+bu | komfo\s+anokye |
      government\s+(?:hospital|clinic|health) |
      (?:accra|kumasi|tamale|ho|cape\s+coast|bolgatanga|wa|koforidua)\s+(?:regional\s+hospital|teaching|metropolitan) |
      accra\s+psychiatric | effia\s+nkwanta | tamale\s+teaching |
      cape\s+coast\s+teaching | ho\s+teaching | bolgatanga\s+regional
    )\b
    """, re.I
)

_FAITH_RE = re.compile(
    r"""(?xi)
    \b(
      catholic | methodist | presbyterian | SDA\b |
      seventh.?day\s+adventist | baptist | anglican | lutheran |
      christian\s+(?:hospital|health|clinic|service|mission|medical) |
      islamic\s+(?:hospital|medical|clinic) | muslim\s+(?:hospital|clinic) |
      church\s+of\s+(?:christ|pentecost|scotland|god|ghana) |
      faith(?:\s+based)? | salvation\s+army |
      mercy\s+(?:hospital|clinic|health|women) |
      holy\s+(?:family|cross|trinity|rosary|ghost|spirit) |
      sacred\s+heart | our\s+lady | immaculate |
      mission\s+(?:hospital|clinic|health\s+center) |
      CHAG\b | ecwa\b | foursquare | assemblies\s+of\s+god |
      true\s+vine | word\s+of\s+life |
      good\s+samaritan | good\s+shepherd |
      ahmadiyya | mosque\s+(?:hospital|clinic) |
      st\.?\s+(?:luke|mary|joseph|john|peter|michael|francis|george|martin|anne|theresa|augustine|dominic|margaret) |
      saint\s+(?:luke|mary|joseph|john|peter|michael|francis|george|martin|anne|theresa|augustine|dominic|margaret) |
      quaker | brethren | mennonite | seven.?day
    )\b
    """, re.I
)

_NGO_RE = re.compile(
    r"""(?xi)
    \b(
      foundation(?!\s+(?:hospital|clinic|medical|health)) |
      non[.\-]?profit | non[.\-]?governmental | NGO\b |
      charity(?!\s+(?:hospital|clinic|medical|health|trust)) |
      charitable\s+organization | humanitarian |
      volunteer\s+(?:program|health) | medical\s+(?:relief|outreach) |
      partners\s+in\s+health | health\s+for\s+all | right\s+to\s+health |
      (?:international|global)\s+(?:health|medical|relief|aid|development) |
      marie\s+stopes | breast\s+care\s+international |
      freedom\s+aid\s+ghana | focos | west\s+africa\s+aids\s+foundation |
      aids\s+foundation | international\s+health\s+care\s+center
    )\b
    """, re.I
)

_PRIVATE_RE = re.compile(
    r"""(?xi)
    \b(
      Ltd\.?$|Limited$|LLC\b|Inc\.?\b|plc\b|PLC\b|
      private(?:\s+limited)? |
      (?:medical|health)\s+(?:enterprise|venture|group\s+of\s+companies) |
      (?:healthcare|wellness)\s+(?:solutions|group|network) |
      proprietor | specialist\s+(?:hospital|clinic)\b |
      (?:dr|doctor)[\.\s]+\w+['s]*\s+(?:clinic|hospital|practice)
    )\b
    """, re.I
)


@pandas_udf(ArrayType(StringType()))  # returns [category, confidence, source, is_inferred]
def expand_org_category_udf(
    existing: pd.Series, name: pd.Series, desc: pd.Series,
    mission: pd.Series, operator_type: pd.Series,
) -> pd.Series:
    results = []
    for ex, nm, dc, ms, op in zip(existing, name, desc, mission, operator_type):
        ex_str = str(ex or "").strip()
        if ex_str and ex_str.lower() not in ("null", "none", "nan", ""):
            results.append([ex_str, "high", "original", "false"])
            continue

        nm_str = str(nm or "")
        dc_str = str(dc or "")
        ms_str = str(ms or "")
        op_str = str(op or "").strip().lower()

        # Tier 1: name inference (high confidence)
        if _GOVT_RE.search(nm_str):
            results.append(["government",      "high",   "name_inferred",        "true"])
        elif _FAITH_RE.search(nm_str):
            results.append(["faith_based",     "high",   "name_inferred",        "true"])
        elif _NGO_RE.search(nm_str):
            results.append(["ngo",             "medium", "name_inferred",        "true"])
        elif _PRIVATE_RE.search(nm_str):
            results.append(["private", "medium", "name_inferred",        "true"])
        # Tier 2: description + mission (medium confidence)
        elif _GOVT_RE.search(dc_str + " " + ms_str):
            results.append(["government",      "medium", "description_inferred", "true"])
        elif _FAITH_RE.search(dc_str + " " + ms_str):
            results.append(["faith_based",     "medium", "description_inferred", "true"])
        elif _NGO_RE.search(dc_str + " " + ms_str):
            results.append(["ngo",             "medium", "description_inferred", "true"])
        elif _PRIVATE_RE.search(dc_str + " " + ms_str):
            results.append(["private", "low",    "description_inferred", "true"])
        # Tier 3: operatorTypeId fallback
        elif op_str == "public":
            results.append(["government",      "low",    "operatorType_fallback", "true"])
        else:
            # v11: default to private_company only when not clearly public
            results.append(["private", "low",    "operatorType_fallback", "true"])

    return pd.Series(results)


silver = silver.withColumn(
    "_org_expansion",
    expand_org_category_udf(
        F.col("organization_category"), F.col("name"),
        F.col("description"), F.col("missionstatement"),
        F.col("operatorTypeId"),
    )
).withColumn("organization_category",           F.col("_org_expansion").getItem(0)
).withColumn("organization_category_confidence", F.col("_org_expansion").getItem(1)
).withColumn("organization_category_source",     F.col("_org_expansion").getItem(2)
).withColumn("organization_category_inferred",   # v11 new
    F.col("_org_expansion").getItem(3).cast(BooleanType())
).drop("_org_expansion")

# Sync boolean flags
for _bool_col, _cat_val in [
    ("is_ngo",         "ngo"),
    ("is_faith_based", "faith_based"),
    ("is_government",  "government"),
]:
    silver = silver.withColumn(
        _bool_col,
        F.coalesce(
            F.col(_bool_col),
            F.when(F.lower(F.coalesce(F.col("organization_category"), F.lit(""))) == _cat_val, F.lit(True)),
            F.lit(False),
        )
    )

print("Organization category after expansion (v11):")
silver.groupBy("organization_category").count().orderBy(F.desc("count")).show()
print("Confidence breakdown:")
silver.groupBy("organization_category_confidence").count().orderBy(F.desc("count")).show()
_null_org = silver.filter(F.col("organization_category").isNull()).count()
print(f"Still null: {_null_org:,} / {_survivor_count:,}  (target = 0)")

# COMMAND ----------
# MAGIC %md ## 7. Hospital Sub-Classification

# COMMAND ----------

_TEACHING_RE = re.compile(
    r"\b(teaching\s+hospital|university\s+hospital|medical\s+(?:university|school)|"
    r"academic\s+medical\s+cent(?:er|re)|residency\s+training|medical\s+training)\b", re.I
)
_REFERRAL_RE = re.compile(
    r"\b(referral\s+hospital|regional\s+(?:hospital|medical\s+cent(?:er|re))|"
    r"national\s+(?:hospital|medical|referral)|tertiary\s+(?:care|hospital)|"
    r"level\s+[3-5]\s+(?:hospital|facility))\b", re.I
)
_MILITARY_RE = re.compile(
    r"\b(military\s+hospital|armed\s+forces|police\s+(?:hospital|clinic|medical)|"
    r"navy\s+(?:hospital|medical)|air\s+force\s+(?:hospital|medical)|"
    r"\d+BN\s+(?:Military\s+)?Hospital|37\s+(?:Military\s+)?Hospital)\b", re.I
)
_SPECIALIST_RE = re.compile(
    r"\b(specialist\s+(?:hospital|cent(?:er|re)|clinic)|eye\s+hospital|"
    r"cardiothoracic\s+cent(?:er|re)|cancer\s+cent(?:er|re)|"
    r"children[\'s]*\s+hospital|maternity\s+hospital|psychiatric\s+hospital|"
    r"dental\s+hospital|orthopaedic\s+(?:hospital|cent(?:er|re)))\b", re.I
)


@pandas_udf(ArrayType(StringType()))
def hospital_subclass_udf(name: pd.Series, cap: pd.Series, desc: pd.Series, ftype: pd.Series) -> pd.Series:
    result = []
    for nm, ca, dc, ft in zip(name, cap, desc, ftype):
        if ca is None:
            cap_text = ""
        elif isinstance(ca, (list, tuple)):
            cap_text = " ".join(str(x) for x in ca)
        else:
            try:
                cap_text = " ".join(str(x) for x in list(ca))
            except:
                cap_text = str(ca)
        combined = " ".join([str(nm or ""), cap_text, str(dc or "")])
        result.append([
            str(bool(_TEACHING_RE.search(combined))),
            str(bool(_REFERRAL_RE.search(combined))),
            str(bool(_MILITARY_RE.search(combined))),
            str(bool(_SPECIALIST_RE.search(combined))),
        ])
    return pd.Series(result)


silver = (
    silver.withColumn("_hsc",
        hospital_subclass_udf(
            F.col("name"), F.col("capability_parsed"),
            F.col("description"), F.col("facility_type_clean")
        ))
    .withColumn("is_teaching_hospital",   F.col("_hsc").getItem(0).cast(BooleanType()))
    .withColumn("is_referral_center",     F.col("_hsc").getItem(1).cast(BooleanType()))
    .withColumn("is_military_hospital",   F.col("_hsc").getItem(2).cast(BooleanType()))
    .withColumn("is_specialist_hospital", F.col("_hsc").getItem(3).cast(BooleanType()))
    .drop("_hsc")
)

print("Hospital sub-classification:")
for _f in ["is_teaching_hospital", "is_referral_center", "is_military_hospital", "is_specialist_hospital"]:
    print(f"  {_f}: {silver.filter(F.col(_f)).count()}")

# COMMAND ----------
# MAGIC %md ## 8. Physical Address Detection (v11)

# COMMAND ----------

_PHYSICAL_RE = re.compile(
    r"""(?xi)
    (
      \b(?:road|rd\.?|street|st\.?|avenue|ave\.?|lane|drive|dr\.?|
          highway|hwy|motorway|bypass|ring\s+road|close|link|crescent|circle)\b
      | \b(?:no\.?\s*\d+|house\s+no\.?\s*\d+|plot\s+(?:no\.?\s*)?\d+|
            block\s+[A-Z0-9]+|unit\s+\d+|flat\s+\d+)\b
      | ^\s*\d{1,4}[\s/,]
      | \b(?:opp(?:osite)?\.?|near\s+|beside\s+|adjacent\s+to\s+|
            behind\s+|next\s+to\s+|junction\s+(?:of|with)|
            close\s+to\s+|at\s+the\s+|off\s+the\s+)\b
      | \b(?:compound\b|estate\s+(?:road|avenue|street)?|
            residential\s+area|housing(?:\s+estate|\s+down)?\b|
            industrial\s+area|trading\s+area|market\s+area|
            suburb|crescent|circle|camp\b)\b
      | \b[A-Z]{2}[-\s]?\d{3,4}[-\s]?\d{4}\b
      | \bMain\s+Street\b | \bMain\s+Road\b | \bJunction\b | \bRoundabout\b
      | \b(?:spintex|lapaz|madina|dansoman|kaneshie|achimota|east\s+legon|
            ringway|liberation|independence\s+avenue|castle\s+road|
            airport\s+(?:city|residential)|north\s+industrial|
            tema\s+community|ridge)\b
      | \bRingway\s+Estate\b | \bTantra\s+Hill\b | \bEast\s+Legon\s+Hills\b
    )
    """, re.I
)

_PO_BOX_ONLY_RE = re.compile(
    r"^\s*(?:p\.?\s*o\.?\s*box|pmb|private\s+mail\s+bag)\s+[\w\-]+\s*$", re.I
)


@pandas_udf(BooleanType())
def fix_physical_address_udf(
    a1: pd.Series, a2: pd.Series, a3: pd.Series, existing: pd.Series
) -> pd.Series:
    result = []
    for a1v, a2v, a3v, ex in zip(a1, a2, a3, existing):
        if ex is True:
            result.append(True)
            continue
        combined = " ".join(str(x or "") for x in [a1v, a2v, a3v]).strip()
        if not combined or combined.lower() in ("null", "none", "nan"):
            result.append(False)
            continue
        if _PO_BOX_ONLY_RE.match(combined):
            result.append(False)
            continue
        result.append(bool(_PHYSICAL_RE.search(combined)))
    return pd.Series(result)


gold = silver.withColumn(
    "has_physical_address",
    fix_physical_address_udf(
        F.col("address_line1"), F.col("address_line2"),
        F.col("address_line3"), F.col("has_physical_address"),
    )
)

_phys_before = silver.filter(F.col("has_physical_address")).count()
_phys_after  = gold.filter(F.col("has_physical_address")).count()
print(f"has_physical_address: before={_phys_before:,}  after={_phys_after:,}  (+{_phys_after - _phys_before})")

# COMMAND ----------
# MAGIC %md ## 9. Second-Pass Clinical Array Cleaning + Recompute Counts

# COMMAND ----------

_GOLD_JUNK_RE = re.compile(
    r"("
    r"has\s+a\s+telephone\s+number\s+at|has\s+(a\s+)?phone\s+number"
    r"|has\s+(an?\s+)?email\s+address|has\s+a\s+location\s+at|has\s+a\s+website\s+at"
    r"|^\d[\d\s\-\+\(\)]*$"
    r"|^\d+\s+(?:people\s+)?(?:like|follow|check.?in)"
    r"|\brating\s*[:\-]?\s*[\d\.]+"
    r"|^\d+\s+reviews?"
    r"|\bis\s+(?:privately|government|publicly)\s+owned\b"
    r"|\bFacility\s+type\s*[:\-]|\bType\s*[:\-]\s*(?:Primary|Secondary|Tertiary)"
    r"|\bNHIS\s+(?:accredited|registered)\b"
    r"|http[s]?://|www\."
    r"|facebook\b|instagram\b|twitter\b"
    r"|^24/7$|^24\s+hours$|^always\s+open$"
    r"|we\s+should\s+not\s+make|wait\s*[\-–]\s*we\s+should"
    r")",
    re.I
)

_MIN_STRIPPED_LEN = 8


def _gold_clean_array(arr):
    if not arr:
        return []
    seen, out = set(), []
    for v in arr:
        if not v:
            continue
        s = str(v).strip()
        stripped = re.sub(r"[^\w\s]", "", s)
        if len(stripped.strip()) < _MIN_STRIPPED_LEN:
            continue
        if _GOLD_JUNK_RE.search(s):
            continue
        key = re.sub(r"\s+", " ", stripped.lower()).strip()
        if key not in seen:
            seen.add(key)
            out.append(s)
    return out


_gold_clean_udf = F.udf(_gold_clean_array, ArrayType(StringType()))

for _arr_col in ("procedure_parsed", "equipment_parsed", "capability_parsed", "specialties_parsed"):
    gold = gold.withColumn(_arr_col, _gold_clean_udf(F.col(_arr_col)))

gold = (
    gold
    .withColumn("procedure_count",  F.size(F.col("procedure_parsed")))
    .withColumn("equipment_count",  F.size(F.col("equipment_parsed")))
    .withColumn("capability_count", F.size(F.col("capability_parsed")))
    .withColumn("specialty_count",  F.size(F.col("specialties_parsed")))
    .withColumn("has_procedures",   F.col("procedure_count")  > 0)
    .withColumn("has_equipment",    F.col("equipment_count")  > 0)
    .withColumn("has_capabilities", F.col("capability_count") > 0)
    .withColumn("has_specialties",  F.col("specialty_count")  > 0)
)

print("Array counts after second-pass cleaning:")
for _c in ("procedure_count", "equipment_count", "capability_count", "specialty_count"):
    _avg = gold.agg(F.avg(_c)).collect()[0][0] or 0
    _max = gold.agg(F.max(_c)).collect()[0][0] or 0
    _nz  = gold.filter(F.col(_c) > 0).count()
    print(f"  {_c:<22}  avg={_avg:.2f}  max={_max}  non-zero={_nz}")

# COMMAND ----------
# MAGIC %md ## 10. phone_numbers STRING → ARRAY<STRING>

# COMMAND ----------

gold = gold.withColumn(
    "phone_numbers",
    F.when(F.col("phone_numbers_parsed").isNotNull(), F.col("phone_numbers_parsed")).otherwise(F.array()),
)

# v11: phone count helper
gold = gold.withColumn("phone_count", F.size(F.col("phone_numbers")).cast(IntegerType()))
gold = gold.withColumn("has_multiple_phones", F.col("phone_count") > 1)

# COMMAND ----------
# MAGIC %md ## 11. Geo Quality Scoring
# MAGIC
# MAGIC v11-F1: geopy_nominatim upgraded to tier 5 (most precise); tier 3 now = city dict.

# COMMAND ----------

gold = gold.withColumn(
    "geo_precision_tier",
    F.when(F.col("geo_source") == "geopy_nominatim",                              F.lit(5))  # v11: most precise
     .when(F.col("geo_source").isin("text_extracted_city"),                        F.lit(4))
     .when(F.col("geo_source").isin("static_city_dict", "extended_city_dict"),     F.lit(3))  # v11: was 4
     .when(F.col("geo_source") == "district_centroid",                             F.lit(3))
     .when(F.col("geo_source") == "region_centroid",                               F.lit(2))
     .when(F.col("geo_source") == "country_centroid",                              F.lit(1))
     .otherwise(F.lit(0))
     .cast(IntegerType())
)

gold = gold.withColumn(
    "geo_quality_score",
    F.when(F.col("geo_source") == "geopy_nominatim",                               F.lit(0.98))
     .when(F.col("geo_source") == "text_extracted_city",                           F.lit(0.90))
     .when(F.col("geo_source").isin("static_city_dict", "extended_city_dict"),     F.lit(0.82))
     .when(F.col("geo_source") == "district_centroid",                             F.lit(0.70))
     .when(F.col("geo_source") == "region_centroid",                               F.lit(0.40))
     .when(F.col("geo_source") == "country_centroid",                              F.lit(0.10))
     .otherwise(F.lit(0.0))
     .cast(FloatType())
)

# Geo contradiction: country_centroid + decent completeness + has contact → suspicious
gold = gold.withColumn(
    "geo_contradiction_flag",
    F.when(
        (F.col("geo_source") == "country_centroid") &
        (F.col("data_completeness_score") >= 0.5) &
        F.col("has_contact"),
        F.lit(True)
    ).otherwise(F.lit(False))
)

print("Geo quality distribution (v11 — revised tiers):")
gold.groupBy("geo_source", "geo_precision_tier").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 12. Evidence Weighting

# COMMAND ----------

_ew_map = F.create_map(*[F.lit(x) for pair in EVIDENCE_WEIGHT_MAP.items() for x in pair])
gold = gold.withColumn(
    "evidence_weight",
    F.coalesce(_ew_map[F.col("source_trust")], F.lit(0.50)).cast(FloatType())
)

# COMMAND ----------
# MAGIC %md ## 13. Save specialty_direct_count (before transitive enrichment)

# COMMAND ----------

gold = gold.withColumn("specialty_direct_count",
    F.size(F.col("specialties_parsed")).cast(IntegerType())
)

# COMMAND ----------
# MAGIC %md ## 14. Transitive Clinical Inference (v11 — BUG FIXED)
# MAGIC
# MAGIC CRITICAL FIX B1: In v10 the `gold = gold.withColumn(...)` application was
# MAGIC erroneously placed INSIDE the UDF function body due to indentation error.
# MAGIC The UDF was defined but the enrichment was NEVER applied (has_icu stayed at 1.7%).
# MAGIC Here we define the function and apply it as two separate, correctly-indented blocks.

# COMMAND ----------

_FDR_VALID_SPECIALTIES = frozenset({
    "internalMedicine", "familyMedicine", "pediatrics", "cardiology",
    "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics",
    "orthopedicSurgery", "dentistry", "ophthalmology", "radiology",
    "pathology", "infectiousDiseases", "nephrology", "criticalCareMedicine",
    "cardiacSurgery", "plasticSurgery", "neurology", "psychiatry",
    "anesthesia", "dermatology", "urology", "gastroenterology",
    "pulmonology", "endocrinologyAndDiabetesAndMetabolism",
    "neonatologyPerinatalMedicine", "medicalOncology",
    "physicalMedicineAndRehabilitation", "otolaryngology",
    "geriatricsInternalMedicine", "hospiceAndPalliativeInternalMedicine",
    "publicHealth", "globalHealthAndInternationalHealth",
    "clinicalPathology", "obstetricsAndMaternityCare",
    "reproductiveEndocrinologyAndInfertility",
    "maternalFetalMedicineOrPerinatology", "socialAndBehavioralSciences",
    "orthodontics", "familyPlanningAndComplexContraception",
})


def _safe_list(v):
    if v is None:
        return []
    if isinstance(v, (list, tuple)):
        return list(v)
    if isinstance(v, np.ndarray):
        return v.tolist()
    try:
        return list(v)
    except:
        return [str(v)] if v else []


# ── UDF DEFINITION ──────────────────────────────────────────────────────────
@pandas_udf(ArrayType(StringType()))
def transitive_enrich_specialties_udf(
    specialties:   pd.Series,
    procedures:    pd.Series,
    equipment:     pd.Series,
    capabilities:  pd.Series,
    is_teaching:   pd.Series,
    is_referral:   pd.Series,
    facility_type: pd.Series,
    description:   pd.Series,
) -> pd.Series:
    results = []
    for sp, pr, eq, cp, teach, refer, ftype, desc in zip(
        specialties, procedures, equipment,
        capabilities, is_teaching, is_referral,
        facility_type, description
    ):
        sp_list = [str(x) for x in _safe_list(sp)]
        pr_list = [str(x) for x in _safe_list(pr)]
        eq_list = [str(x) for x in _safe_list(eq)]
        cp_list = [str(x) for x in _safe_list(cp)]

        spec_set = set(sp_list)
        text_parts = sp_list + pr_list + eq_list + cp_list
        if isinstance(desc, str) and desc.strip():
            text_parts.append(desc)
        combined = " ".join(text_parts).lower()
        ftype_str = str(ftype or "").strip().lower()

        # ── Facility-type baseline ───────────────────────────────────────
        if teach or refer or ftype_str in ("teaching hospital", "regional hospital", "tertiary hospital"):
            for s in ("internalMedicine", "generalSurgery", "pediatrics",
                      "gynecologyAndObstetrics", "radiology", "pathology",
                      "emergencyMedicine", "anesthesia", "criticalCareMedicine"):
                spec_set.add(s)

        if ftype_str in ("district hospital", "secondary hospital"):
            spec_set.update(("internalMedicine", "generalSurgery",
                             "gynecologyAndObstetrics", "pediatrics", "emergencyMedicine"))

        if ftype_str in ("eye clinic", "eye_clinic"):
            spec_set.add("ophthalmology")

        if ftype_str in ("dentist", "dental clinic"):
            spec_set.add("dentistry")

        if ftype_str in ("maternity home", "maternity_home"):
            spec_set.update(("gynecologyAndObstetrics", "obstetricsAndMaternityCare"))

        if ftype_str in ("diagnostic center", "diagnostic_centre"):
            spec_set.update(("radiology", "clinicalPathology"))

        if ftype_str in ("chps", "community health post"):
            # v11-F8: CHPS gets familyMedicine + publicHealth
            spec_set.update(("familyMedicine", "publicHealth"))

        # ── Keyword-driven inference ─────────────────────────────────────

        # Obstetrics / maternity
        if any(kw in combined for kw in (
            "caesarean", "c-section", "c section", "maternity", "labour and delivery",
            "labor and delivery", "antenatal care", "delivery room", "labour ward",
            "labor ward", "midwif", "obstetric", "gynecology", "gynaecology",
            "fistula repair", "prenatal", "postnatal",
        )):
            spec_set.add("gynecologyAndObstetrics")
            spec_set.add("obstetricsAndMaternityCare")

        # Critical care / ICU — tightened in v11 (require specific ICU terms)
        if any(kw in combined for kw in (
            "ventilator", "mechanical ventilat", "life support",
            "intensive care unit", "intensive care bed",
            " icu ", "icu bed", "hdu", "high dependency unit",
            "crash cart", "picu", "neonatal intensive care unit", "nicu",
        )):
            spec_set.add("criticalCareMedicine")

        # Emergency medicine
        if any(kw in combined for kw in (
            "defibrillator", "resuscitat", "trauma center", "trauma care",
            "emergency department", "emergency room", "accident and emergency",
            "a&e", "emergency ward", "first response", "emergency services",
            "casualty department", "24 hour emergency",
        )):
            spec_set.add("emergencyMedicine")

        # Dialysis / nephrology
        if any(kw in combined for kw in (
            "dialysis", "hemodialysis", "haemodialysis",
        )):
            spec_set.add("nephrology")

        # Radiology / imaging
        if any(kw in combined for kw in (
            "x-ray machine", "ct scanner", "mri machine", "mri scanner",
            "mammograph", "fluoroscopy", "ultrasound machine",
            "ct scan", "mri scan", "radiology department", "imaging department",
        )):
            spec_set.add("radiology")

        # Cardiology
        if any(kw in combined for kw in (
            "ecg machine", "electrocardiog", "echocardiograph", "holter monitor",
            "cardiac catheter", "angiogram", "cardiolog", "cardiac surgery",
            "pacemaker",
        )):
            spec_set.add("cardiology")
            if "cardiac surgery" in combined or "heart surgery" in combined:
                spec_set.add("cardiacSurgery")

        # General surgery
        if any(kw in combined for kw in (
            "operating theatre", "operating room", "anaesthesia machine",
            "anesthesia machine", "surgical suite", "laparoscope",
            "appendectomy", "hernia repair", "surgical procedure", "surgery",
            "minor surgery", "major surgery",
        )):
            spec_set.add("generalSurgery")
            spec_set.add("anesthesia")

        # Ophthalmology
        if any(kw in combined for kw in (
            "slit lamp", "tonometer", "fundus camera", "phoropter",
            "cataract", "glaucoma surgery", "eye surgery", "ophthalmo",
            "optometrist", "eye care", "retina", "cornea",
        )):
            spec_set.add("ophthalmology")

        # Dentistry
        if any(kw in combined for kw in (
            "dental chair", "dental unit", "tooth extraction", "dental x",
            "dental care", "dentist", "orthodont", "oral health",
        )):
            spec_set.add("dentistry")

        # Pediatrics / neonatology
        if any(kw in combined for kw in (
            "neonatal incubator", "infant warmer", "neonatal care unit",
            "child health", "pediatric", "paediatric", "child clinic",
            "children's ward", "child welfare",
        )):
            spec_set.add("neonatologyPerinatalMedicine")
            spec_set.add("pediatrics")

        # Infectious disease
        if any(kw in combined for kw in (
            "antiretroviral", "art clinic", "hiv treatment", "pmtct",
            "tb treatment", "isolation ward", "infectious disease",
            "tropical disease", "malaria", "leprosy", "hepatitis", "covid",
        )):
            spec_set.add("infectiousDiseases")

        # Psychiatry
        if any(kw in combined for kw in (
            "psychiatric", "mental health", "psycholog", "counsel",
            "behavio?ral therapy", "psychotherapy", "electroconvulsive",
        )):
            spec_set.add("psychiatry")

        # Orthopedic
        if any(kw in combined for kw in (
            "orthopaedic", "orthopedic", "fracture", "bone surgery",
            "joint replacement", "spine surgery", "prosthetic",
        )):
            spec_set.add("orthopedicSurgery")

        # Urology
        if any(kw in combined for kw in (
            "urology", "prostate", "urinary tract", "kidney stone", "cystoscop",
        )):
            spec_set.add("urology")

        # Dermatology
        if any(kw in combined for kw in (
            "dermatolog", "skin care", "skin condition", "eczema",
        )):
            spec_set.add("dermatology")

        # Gastroenterology
        if any(kw in combined for kw in (
            "gastroenterolog", "colonoscop", "endoscop", "digestive disease",
        )):
            spec_set.add("gastroenterology")

        # Pulmonology
        if any(kw in combined for kw in (
            "respiratory", "pulmonar", "spirometer", "asthma", "copd",
        )):
            spec_set.add("pulmonology")

        # Endocrinology
        if any(kw in combined for kw in (
            "diabetes", "endocrin", "thyroid", "hormone", "insulin",
        )):
            spec_set.add("endocrinologyAndDiabetesAndMetabolism")

        # Oncology
        if any(kw in combined for kw in (
            "oncology", "chemotherapy", "radiotherapy", "cancer",
        )):
            spec_set.add("medicalOncology")

        # Clinical pathology / lab
        if any(kw in combined for kw in (
            "laboratory", "lab test", "haematology", "hematology",
            "biochemistry", "microbiology", "pathology", "blood bank",
        )):
            spec_set.add("clinicalPathology")

        # ENT
        if any(kw in combined for kw in (
            "ear nose throat", "ent ", " otorhinolaryng", "audiology",
        )):
            spec_set.add("otolaryngology")

        # Physical medicine
        if any(kw in combined for kw in (
            "physiotherapy", "physical therapy", "rehabilitation",
            "occupational therapy",
        )):
            spec_set.add("physicalMedicineAndRehabilitation")

        # Palliative
        if any(kw in combined for kw in (
            "palliative", "hospice", "end of life",
        )):
            spec_set.add("hospiceAndPalliativeInternalMedicine")

        # Public health
        if any(kw in combined for kw in (
            "public health", "community health", "immunization",
            "vaccination", "outreach program",
        )):
            spec_set.add("publicHealth")

        # Reproductive
        if any(kw in combined for kw in (
            "ivf", "in vitro", "fertility", "infertility",
        )):
            spec_set.add("reproductiveEndocrinologyAndInfertility")

        # Family planning
        if any(kw in combined for kw in (
            "family planning", "contraception", "birth control",
        )):
            spec_set.add("familyPlanningAndComplexContraception")

        # ── Filter to FDR valid taxonomy ─────────────────────────────────
        valid_specs = sorted(s for s in spec_set if s in _FDR_VALID_SPECIALTIES)

        # Fallback: internalMedicine unless clearly non-clinical
        if not valid_specs and ftype_str not in ("pharmacy", "alternative medicine"):
            valid_specs = ["internalMedicine"]

        results.append(valid_specs)

    return pd.Series(results)


# ── FIX B1: UDF APPLICATION IS NOW CORRECTLY OUTSIDE THE FUNCTION BODY ─────
gold = gold.withColumn(
    "specialties_parsed",
    transitive_enrich_specialties_udf(
        F.col("specialties_parsed"), F.col("procedure_parsed"),
        F.col("equipment_parsed"),   F.col("capability_parsed"),
        F.col("is_teaching_hospital"), F.col("is_referral_center"),
        F.col("facility_type_clean"),
        F.col("description"),
    )
).withColumn("specialty_count", F.size(F.col("specialties_parsed")))

print("Specialty stats after transitive inference (v11 — bug fixed):")
gold.agg(
    F.round(F.avg("specialty_count"), 2).alias("avg"),
    F.max("specialty_count").alias("max"),
    F.sum(F.when(F.col("specialty_count") > 0, 1).otherwise(0)).alias("non_zero"),
).show()

# COMMAND ----------
# MAGIC %md ## 15. Specialty Presence Flags (v11)

# COMMAND ----------

SPEC_TAXONOMY = {
    "has_emergency_medicine": [
        "emergencyMedicine", "emergencyPreparednessAndDisasterResponse",
        "accidentAndEmergency", "urgentCare", "traumaSurgery",
    ],
    "has_obstetrics": [
        "gynecologyAndObstetrics", "obstetricsAndMaternityCare",
        "maternalFetalMedicineOrPerinatology", "familyPlanningAndComplexContraception",
        "maternalAndChildHealth", "midwifery", "antenatalCare", "postnatalCare",
        "obstetrics",
    ],
    "has_surgery": [
        "generalSurgery", "orthopedicSurgery", "cardiacSurgery", "plasticSurgery",
        "neurosurgery", "pediatricSurgery", "vascularSurgery",
        "oralAndMaxillofacialSurgery", "craniofacialSurgery",
        "surgicalOncology", "colorectalSurgery",
    ],
    "has_pediatrics": [
        "pediatrics", "neonatologyPerinatalMedicine", "pediatricCardiology",
        "pediatricSurgery", "adolescentMedicine",
        "childHealth", "paediatrics", "newbornMedicine",
    ],
    "has_icu": [
        "criticalCareMedicine", "intensiveCareMedicine", "criticalCare",
        "neurocriticalCare", "surgicalCriticalCare",
    ],
    "has_radiology": [
        "radiology", "diagnosticRadiology", "interventionalRadiology",
        "breastImaging", "nuclearMedicineAndMolecularImaging",
    ],
    "has_infectious_disease": [
        "infectiousDiseases", "communicableDisease",
        "tropicalMedicine", "hivMedicine",
    ],
    "has_mental_health": [
        "psychiatry", "addictionPsychiatry", "childAndAdolescentPsychiatry",
        "socialAndBehavioralSciences", "clinicalPsychology", "psychiatricNursing",
    ],
}

TEXT_KEYWORDS = {
    "has_emergency_medicine": [
        "emergency department", "emergency room", "emergency unit", "emergency ward",
        "emergency care", "accident and emergency", "a&e", "casualty department",
        "casualty ward", "trauma center", "trauma care", "trauma unit",
        "resuscitation", "urgent care", "24/7 emergency", "triage unit",
        "emergency medical", "first response", "24 hour emergency",
        "accident & emergency", "a and e", "er physician",
    ],
    "has_obstetrics": [
        "antenatal", "prenatal care", "postnatal care", "labour ward", "labor ward",
        "delivery room", "maternity ward", "maternity home", "maternity unit",
        "maternity services", "caesarean", "c-section", "midwifery",
        "obstetric care", "gynecology services", "gynaecology services",
        "maternal health", "safe motherhood", "family planning", "reproductive health",
        "ob gyn", "obgyn", "fetal assessment", "anc", "pnc",
    ],
    "has_surgery": [
        "operating theatre", "operating room", "surgical procedures",
        "laparoscopy", "laparoscopic surgery", "appendectomy", "hernia repair",
        "cholecystectomy", "surgical suite", "general surgery",
        "orthopaedic surgery", "orthopedic surgery", "cardiac surgery",
        "surgical ward", "theatre", "surgical unit", "laparotomy",
    ],
    "has_pediatrics": [
        "neonatal care", "nicu", "neonatal intensive care", "child health",
        "paediatric ward", "pediatric ward", "infant care", "newborn care",
        "child and adolescent", "children ward", "paediatric", "pediatric services",
        "well-baby", "childhood immunisation", "pediatrician",
    ],
    "has_icu": [
        "intensive care unit", "intensive care", "critical care unit",
        "high dependency unit", "hdu", "mechanical ventilation", "ventilator support",
        "life support", "neonatal intensive care unit", "nicu beds", "picu",
        "critical care ward", "icu beds", "icu", "intensive care bed",
        "ventilator", "critical care",
    ],
    "has_radiology": [
        "x-ray", "x ray", "ultrasound scan", "ct scan", "mri scan",
        "mammography", "fluoroscopy", "imaging center", "diagnostic imaging",
        "medical imaging", "radiography", "sonography", "digital x-ray",
        "ultrasound", "ct scanner", "mri scanner",
    ],
    "has_infectious_disease": [
        "hiv treatment", "aids care", "hiv/aids", "antiretroviral therapy",
        "art clinic", "tuberculosis treatment", "tb clinic", "malaria treatment",
        "pmtct", "infection control", "isolation ward", "communicable disease",
        "infectious disease", "hepatitis b", "hiv positive",
    ],
    "has_mental_health": [
        "mental health services", "psychiatric ward", "psychiatric hospital",
        "psychiatry unit", "psychology services", "psychotherapy",
        "counseling services", "counselling services", "mental health clinic",
        "addiction treatment", "electroconvulsive therapy",
        "psychiatric", "mental health", "psychologist",
    ],
}


def _clean_text_for_flag(text):
    if not text:
        return ""
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\+?\d[\d\s-]{7,}\d', '', text)
    text = re.sub(r'[\w\.-]+@[\w\.-]+', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def _facility_type_hard_rules(ftype):
    """Hard rules by facility type — always applied regardless of text evidence."""
    rules = {}
    ftype = str(ftype).lower() if ftype else ""
    if ftype in ("maternity_home", "maternity home"):
        rules["has_obstetrics"] = True
    if "psychiatric" in ftype or ftype == "mental_health":
        rules["has_mental_health"] = True
    if ftype in ("diagnostic_center", "diagnostic center"):
        rules["has_radiology"] = True
    return rules


RESULT_SCHEMA = StructType([
    StructField("has_emergency_medicine", BooleanType(), False),
    StructField("has_obstetrics",         BooleanType(), False),
    StructField("has_surgery",            BooleanType(), False),
    StructField("has_pediatrics",         BooleanType(), False),
    StructField("has_icu",                BooleanType(), False),
    StructField("has_radiology",          BooleanType(), False),
    StructField("has_infectious_disease", BooleanType(), False),
    StructField("has_mental_health",      BooleanType(), False),
])


@pandas_udf(returnType=RESULT_SCHEMA)
def compute_all_flags(
    specs: pd.Series, caps: pd.Series, procs: pd.Series,
    equips: pd.Series, desc: pd.Series, facility_type: pd.Series,
    is_teaching: pd.Series, is_referral: pd.Series,
) -> pd.DataFrame:
    tax_sets = {flag: set(s.lower() for s in tax_list) for flag, tax_list in SPEC_TAXONOMY.items()}
    kw_lists = {flag: [k.lower() for k in kw_list] for flag, kw_list in TEXT_KEYWORDS.items()}

    result_rows = []
    for sp, ca, pr, eq, dc, ftype, teach, refer in zip(
        specs, caps, procs, equips, desc, facility_type, is_teaching, is_referral
    ):
        def to_list(val):
            if val is None:
                return []
            if isinstance(val, np.ndarray):   # FIX B2: numpy was not imported — now it is
                return val.tolist()
            if isinstance(val, (list, tuple)):
                return list(val)
            try:
                return list(val)
            except:
                return []

        sp_list    = [str(s).lower() for s in to_list(sp)]
        cap_list   = to_list(ca)
        proc_list  = to_list(pr)
        equip_list = to_list(eq)
        ftype_val  = str(ftype) if ftype is not None else ""

        flags = _facility_type_hard_rules(ftype_val)

        # v11: teaching/referral hospitals always get ICU + emergency + surgery
        if teach or refer:
            flags["has_icu"]                = True
            flags["has_emergency_medicine"] = True
            flags["has_surgery"]            = True

        cap_text  = _clean_text_for_flag(" ".join(str(x) for x in cap_list))
        proc_text = _clean_text_for_flag(" ".join(str(x) for x in proc_list))
        equip_text= _clean_text_for_flag(" ".join(str(x) for x in equip_list))
        desc_text = _clean_text_for_flag(str(dc) if dc else "")
        combined  = " ".join([cap_text, proc_text, equip_text, desc_text])

        for flag, tax_set in tax_sets.items():
            if flags.get(flag, False):
                continue
            if any(s in tax_set for s in sp_list):
                flags[flag] = True
                continue
            if any(kw in combined for kw in kw_lists[flag]):
                flags[flag] = True

        for f in RESULT_SCHEMA.fieldNames():
            flags.setdefault(f, False)

        result_rows.append(tuple(flags[f] for f in RESULT_SCHEMA.fieldNames()))

    return pd.DataFrame(result_rows, columns=RESULT_SCHEMA.fieldNames())


# Apply the UDF
gold = gold.withColumn(
    "spec_struct",
    compute_all_flags(
        F.col("specialties_parsed"), F.col("capability_parsed"),
        F.col("procedure_parsed"),   F.col("equipment_parsed"),
        F.col("description"),        F.col("facility_type_clean"),
        F.col("is_teaching_hospital"), F.col("is_referral_center"),
    )
)

for fname in RESULT_SCHEMA.fieldNames():
    gold = gold.withColumn(fname, F.col(f"spec_struct.{fname}"))
gold = gold.drop("spec_struct")

_tot = gold.count()
print(f"Specialty flags ({_tot:,} rows) — v11 (transitive enrichment applied + teaching→ICU fix):")
for fname in RESULT_SCHEMA.fieldNames():
    n = gold.filter(F.col(fname)).count()
    print(f"  {fname:<34}  {n:>6} / {_tot:,}  ({n/_tot*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 16. Specialty Inferred Count

# COMMAND ----------

gold = gold.withColumn(
    "specialty_inferred_count",
    F.greatest(
        F.lit(0),
        (F.col("specialty_count") - F.col("specialty_direct_count")).cast(IntegerType())
    )
)

# COMMAND ----------
# MAGIC %md ## 17. Classification Booleans + Ownership Model

# COMMAND ----------

gold = (
    gold
    .withColumn("is_public",
        F.when(F.lower(F.trim(F.col("operatorTypeId"))) == "public",   True).otherwise(False))
    .withColumn("is_private",
        F.when(F.lower(F.trim(F.col("operatorTypeId"))) == "private",  True).otherwise(False))
    .withColumn("is_hospital",
        F.when(F.lower(F.coalesce(F.col("facility_type_clean"), F.lit(""))) == "hospital", True).otherwise(False))
    .withColumn("is_clinic",
        F.when(
            F.lower(F.coalesce(F.col("facility_type_clean"), F.lit(""))).isin(
                "clinic", "eye_clinic", "maternity_home", "chps"
            ), True,
        ).otherwise(False))
)

gold = gold.withColumn(
    "ownership_model",
    F.when(F.col("is_military_hospital"),                 F.lit("military"))
     .when(F.col("is_teaching_hospital") & F.col("is_government"), F.lit("academic_government"))
     .when(F.col("is_teaching_hospital"),                 F.lit("academic"))
     .when(F.col("is_ngo") & F.col("is_faith_based"),     F.lit("faith_ngo"))
     .when(F.col("is_ngo"),                               F.lit("ngo"))
     .when(F.col("is_faith_based"),                       F.lit("faith_based"))
     .when(F.col("is_government") | (F.lower(F.coalesce(F.col("organization_category"), F.lit(""))) == "government"),
           F.lit("government"))
     .when(F.col("is_public"),                            F.lit("public"))
     .when(F.lower(F.coalesce(F.col("organization_category"), F.lit(""))) == "private",
           F.lit("private_enterprise"))
     .when(F.col("is_private"),                           F.lit("private"))
     .otherwise(F.lit("unknown"))
)

# v11-F5: Facility tier label — human-readable for RAG queries
gold = gold.withColumn(
    "facility_tier_label",
    F.when(F.col("is_teaching_hospital"),                          F.lit("Academic/Teaching"))
     .when(F.col("is_referral_center"),                            F.lit("Referral/Regional"))
     .when(F.col("is_specialist_hospital"),                        F.lit("Specialist"))
     .when(F.col("is_military_hospital"),                          F.lit("Military"))
     .when(F.col("is_hospital") & F.col("has_icu") & F.col("has_surgery"), F.lit("Secondary+"))
     .when(F.col("is_hospital"),                                   F.lit("Secondary"))
     .when(F.lower(F.coalesce(F.col("facility_type_clean"), F.lit(""))) == "chps",   F.lit("CHPS"))
     .when(F.lower(F.coalesce(F.col("facility_type_clean"), F.lit(""))) == "maternity_home", F.lit("Maternity"))
     .when(F.col("is_clinic"),                                     F.lit("Primary/Clinic"))
     .otherwise(F.lit("Other"))
)

print("Ownership model distribution:")
gold.groupBy("ownership_model").count().orderBy(F.desc("count")).show()
print("Facility tier label distribution:")
gold.groupBy("facility_tier_label").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 18. NGO Ghana Filter

# COMMAND ----------

gold = gold.withColumn(
    "ngo_serves_ghana",
    F.when(
        F.col("is_ngo") & (
            F.array_contains(F.col("countries_parsed"), "GH")
            | F.array_contains(F.col("countries_parsed"), "Ghana")
            | (F.lower(F.trim(F.col("address_countryCode"))) == "gh")
            | F.lower(F.coalesce(F.col("address_country"), F.lit(""))).contains("ghana")
            | (
                (F.size(F.col("countries_parsed")) == 0)
                & F.col("latitude").isNotNull()
                & F.col("latitude").between(4.0, 12.0)
                & F.col("longitude").between(-4.0, 2.0)
            )
        ),
        True,
    ).otherwise(False)
)

print(f"ngo_serves_ghana: {gold.filter(F.col('ngo_serves_ghana')).count()}")

# COMMAND ----------
# MAGIC %md ## 19. Facility Sophistication Model (v11)

# COMMAND ----------

gold = gold.withColumn(
    "clinical_complexity_score",
    F.least(
        F.lit(1.0),
        (
            F.when(F.col("is_hospital"), F.lit(0.10)).otherwise(F.lit(0.0))
            + F.when(F.col("is_clinic"),  F.lit(0.06)).otherwise(F.lit(0.0))
            + F.col("has_icu")               .cast(FloatType()) * 0.25
            + F.col("has_surgery")           .cast(FloatType()) * 0.18
            + F.col("has_emergency_medicine").cast(FloatType()) * 0.12
            + F.col("has_radiology")         .cast(FloatType()) * 0.10
            + F.col("has_obstetrics")        .cast(FloatType()) * 0.08
            + F.col("has_pediatrics")        .cast(FloatType()) * 0.07
            + F.col("has_mental_health")     .cast(FloatType()) * 0.05
            + F.col("has_infectious_disease").cast(FloatType()) * 0.04
            + F.col("is_teaching_hospital") .cast(FloatType()) * 0.12
            + F.col("is_referral_center")   .cast(FloatType()) * 0.08
            + F.col("is_specialist_hospital").cast(FloatType()) * 0.05
            + F.least(F.lit(0.06), F.col("specialty_count").cast(FloatType()) / 35.0)
            + F.least(F.lit(0.04), F.col("equipment_count").cast(FloatType()) / 80.0)
            + F.when(F.col("capacity_int") >= 200, F.lit(0.06))
               .when(F.col("capacity_int") >= 100, F.lit(0.04))
               .when(F.col("capacity_int") >= 30,  F.lit(0.02))
               .when(F.col("capacity_int").isNotNull(), F.lit(0.01))
               .otherwise(F.lit(0.0))
        )
    ).cast(FloatType())
)

gold = gold.withColumn(
    "facility_complexity_level",
    F.when(
        (F.col("is_teaching_hospital") | F.col("is_referral_center"))
        & F.col("has_icu") & F.col("has_surgery"),
        F.lit("L4"),
    ).when(
        (F.col("clinical_complexity_score") >= 0.28) & F.col("is_hospital"),
        F.lit("L3"),
    ).when(
        F.col("is_hospital") | (F.col("is_clinic") & (F.col("clinical_complexity_score") >= 0.08)),
        F.lit("L2"),
    ).otherwise(F.lit("L1"))
)

print("Facility complexity distribution (v11):")
gold.groupBy("facility_complexity_level").count().orderBy("facility_complexity_level").show()

# COMMAND ----------
# MAGIC %md ## 20. Dimensioned Completeness Scores (v11-F2: improved contact completeness)

# COMMAND ----------

gold = gold.withColumn(
    "clinical_completeness",
    (
        F.col("has_specialties") .cast(FloatType()) * 0.20
        + F.col("has_procedures").cast(FloatType()) * 0.25
        + F.col("has_equipment") .cast(FloatType()) * 0.25
        + F.col("has_capabilities").cast(FloatType()) * 0.15
        + F.when(F.col("capacity_int").isNotNull(), F.lit(0.10)).otherwise(F.lit(0.0))
        + F.when(F.col("number_doctors_int").isNotNull(), F.lit(0.05)).otherwise(F.lit(0.0))
    ).cast(FloatType())
)

gold = gold.withColumn(
    "location_completeness",
    (
        F.when(F.col("geo_precision_tier") >= 4, F.lit(0.50))
         .when(F.col("geo_precision_tier") == 3, F.lit(0.38))  # v11: city dict = tier 3 now
         .when(F.col("geo_precision_tier") == 2, F.lit(0.20))
         .otherwise(F.lit(0.05))
        + F.when(F.col("has_physical_address"), F.lit(0.30)).otherwise(F.lit(0.0))
        + F.when(F.col("region_normalised") != "Unknown", F.lit(0.20)).otherwise(F.lit(0.0))
    ).cast(FloatType())
)

# v11-F2: better contact completeness — check phone_numbers array AND official_phone
gold = gold.withColumn(
    "contact_completeness",
    (
        F.when(
            F.col("official_phone").isNotNull() & (F.trim(F.col("official_phone")) != ""),
            F.lit(0.35)
        ).when(
            F.col("phone_count") > 0,
            F.lit(0.25)
        ).otherwise(F.lit(0.0))
        + F.when(
            F.col("email").isNotNull() & (F.trim(F.col("email")) != ""),
            F.lit(0.30)
        ).otherwise(F.lit(0.0))
        + F.when(F.col("officialWebsite").isNotNull(), F.lit(0.20)).otherwise(F.lit(0.0))
        + F.when(F.col("has_multiple_phones"), F.lit(0.15)).otherwise(F.lit(0.0))
    ).cast(FloatType())
)

# COMMAND ----------
# MAGIC %md ## 21. Statistical Anomaly Flags (v11-F10)
# MAGIC
# MAGIC hospital_no_doctors: now EXCLUDES chps and maternity_home (they legitimately may lack doctors).

# COMMAND ----------

gold = gold.withColumn(
    "stat_anomaly_capability_inflation",
    F.when(
        (F.col("capability_count") >= 8)
        & (F.col("procedure_count") == 0)
        & (F.col("equipment_count") == 0)
        & ~F.col("is_ngo"),
        True,
    ).otherwise(False),
)

# v11-F10: Exclude CHPS and maternity homes from hospital_no_doctors
gold = gold.withColumn(
    "stat_anomaly_hospital_no_doctors",
    F.when(
        F.col("is_hospital")
        & F.col("number_doctors_int").isNull()
        & ~F.col("is_ngo")
        & (F.col("source_trust") == "structured")
        & (F.col("data_completeness_score") >= 0.70)
        & (F.col("geo_precision_tier") >= 3)
        & (F.col("has_procedures") | F.col("has_specialties"))
        # v11: CHPS and maternity homes legitimately have no doctors counted
        & ~F.lower(F.coalesce(F.col("facility_type_clean"), F.lit(""))).isin("chps", "maternity_home"),
        True,
    ).otherwise(False),
)

gold = gold.withColumn(
    "stat_anomaly_clinic_claims_icu",
    F.when(F.col("is_clinic") & F.col("has_icu") & ~F.col("is_ngo")
           & ~F.col("is_teaching_hospital") & ~F.col("is_referral_center"),
           True).otherwise(False),
)

gold = gold.withColumn(
    "stat_anomaly_ghost_facility",
    F.when(
        (F.col("data_completeness_score") < 0.45)
        & ~F.col("has_contact")
        & (F.col("capability_count") > 0)
        & (F.col("procedure_count") == 0)
        & (F.col("equipment_count") == 0)
        & ~F.col("is_ngo")
        & F.col("geo_source").isin("country_centroid", "region_centroid"),
        True,
    ).otherwise(False),
)

gold = gold.withColumn(
    "stat_anomaly_specialty_mismatch",
    F.when(
        (F.col("specialty_count") >= 5)
        & (F.col("capability_count") == 0)
        & (F.col("procedure_count") == 0)
        & (F.col("equipment_count") == 0)
        & ~F.col("is_ngo"),
        True,
    ).otherwise(False),
)

gold = gold.withColumn(
    "stat_anomaly_procedure_breadth",
    F.when(
        (F.col("procedure_count") > 8) & (F.col("equipment_count") == 0) & ~F.col("is_ngo"),
        True,
    ).otherwise(False),
)

gold = gold.withColumn(
    "total_stat_anomalies",
    (
        F.col("stat_anomaly_capability_inflation") .cast(IntegerType())
        + F.col("stat_anomaly_hospital_no_doctors")  .cast(IntegerType())
        + F.col("stat_anomaly_clinic_claims_icu")    .cast(IntegerType())
        + F.col("stat_anomaly_ghost_facility")       .cast(IntegerType())
        + F.col("stat_anomaly_specialty_mismatch")   .cast(IntegerType())
        + F.col("stat_anomaly_procedure_breadth")    .cast(IntegerType())
    ),
)

print("Anomaly flags (v11):")
for _f in [
    "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
    "stat_anomaly_clinic_claims_icu",    "stat_anomaly_ghost_facility",
    "stat_anomaly_specialty_mismatch",   "stat_anomaly_procedure_breadth",
]:
    _n = gold.filter(F.col(_f)).count()
    print(f"  {_f:<46}  {_n:>4}  ({_n/_tot*100:.1f}%)")
_any = gold.filter(F.col("total_stat_anomalies") > 0).count()
print(f"  Rows with ≥1 anomaly: {_any:,}  ({_any/_tot*100:.1f}%)")

# COMMAND ----------
# MAGIC %md ## 22. Quality Risk Scores

# COMMAND ----------

gold = gold.withColumn(
    "quality_risk_score",
    F.least(F.lit(1.0), (
        F.when(F.col("data_completeness_score") < 0.35, F.lit(0.30)).otherwise(F.lit(0.0))
        + F.when(F.col("geo_source") == "country_centroid", F.lit(0.20)).otherwise(F.lit(0.0))
        + F.when(F.col("geo_source") == "region_centroid",  F.lit(0.10)).otherwise(F.lit(0.0))
        + F.when(F.col("quality_flag_count") >= 4, F.lit(0.20)).otherwise(
            F.when(F.col("quality_flag_count") >= 2, F.lit(0.10)).otherwise(F.lit(0.0)))
        + F.when(F.col("source_trust") == "social_metadata", F.lit(0.15)).otherwise(F.lit(0.0))
        + F.when(F.col("source_trust") == "inferred",         F.lit(0.08)).otherwise(F.lit(0.0))
        # v11: penalise region mismatch
        + F.when(F.col("geo_region_mismatch"), F.lit(0.05)).otherwise(F.lit(0.0))
    )).cast(FloatType())
)

gold = gold.withColumn(
    "clinical_risk_score",
    F.least(F.lit(1.0), (
        F.when(F.col("stat_anomaly_capability_inflation"), F.lit(0.30)).otherwise(F.lit(0.0))
        + F.when(F.col("stat_anomaly_specialty_mismatch"), F.lit(0.25)).otherwise(F.lit(0.0))
        + F.when(F.col("stat_anomaly_clinic_claims_icu"),  F.lit(0.20)).otherwise(F.lit(0.0))
        + F.when(F.col("stat_anomaly_procedure_breadth"),  F.lit(0.15)).otherwise(F.lit(0.0))
        + F.when(F.col("capability_count") > 15, F.lit(0.10)).otherwise(F.lit(0.0))
    )).cast(FloatType())
)

gold = gold.withColumn(
    "operational_risk_score",
    F.least(F.lit(1.0), (
        F.when(F.col("operational_status") == "closed",   F.lit(0.50)).otherwise(F.lit(0.0))
        + F.when(F.col("operational_status") == "unknown", F.lit(0.20)).otherwise(F.lit(0.0))
        + F.when(F.col("stat_anomaly_ghost_facility"),     F.lit(0.25)).otherwise(F.lit(0.0))
        + F.when(~F.col("has_contact"),                    F.lit(0.15)).otherwise(F.lit(0.0))
        + F.when(~F.col("has_physical_address"),           F.lit(0.10)).otherwise(F.lit(0.0))
    )).cast(FloatType())
)

gold = gold.withColumn(
    "integrity_risk_score",
    F.least(F.lit(1.0), (
        F.when(F.col("source_trust") == "social_metadata",  F.lit(0.25)).otherwise(F.lit(0.0))
        + F.when(F.col("geo_source") == "country_centroid", F.lit(0.20)).otherwise(F.lit(0.0))
        + F.when(F.col("quality_flag_count") >= 3,          F.lit(0.20)).otherwise(F.lit(0.0))
        + F.when(~F.col("has_description"),                  F.lit(0.15)).otherwise(F.lit(0.0))
        + F.when(F.col("evidence_weight") < 0.5,             F.lit(0.10)).otherwise(F.lit(0.0))
        + F.when(F.col("geo_region_mismatch"),               F.lit(0.05)).otherwise(F.lit(0.0))
    )).cast(FloatType())
)

# COMMAND ----------
# MAGIC %md ## 23. Emergency Readiness Score + Critical Care Score

# COMMAND ----------

gold = gold.withColumn(
    "emergency_readiness_score",
    F.least(F.lit(1.0), (
        F.col("has_emergency_medicine").cast(FloatType()) * 0.35
        + F.col("has_icu")            .cast(FloatType()) * 0.25
        + F.col("has_surgery")        .cast(FloatType()) * 0.20
        + F.col("is_referral_center") .cast(FloatType()) * 0.10
        + F.col("is_teaching_hospital").cast(FloatType())* 0.10
        + F.when(F.col("capacity_int") >= 50,  F.lit(0.05)).otherwise(F.lit(0.0))
        + F.when(F.col("number_doctors_int") >= 5, F.lit(0.05)).otherwise(F.lit(0.0))
        - F.when(F.col("geo_source") == "country_centroid", F.lit(0.10)).otherwise(F.lit(0.0))
    )).cast(FloatType())
)

gold = gold.withColumn(
    "critical_care_score",
    F.least(F.lit(1.0), (
        F.col("has_icu")              .cast(FloatType()) * 0.40
        + F.col("has_surgery")        .cast(FloatType()) * 0.30
        + F.col("is_teaching_hospital").cast(FloatType())* 0.15
        + F.col("is_referral_center") .cast(FloatType()) * 0.10
        + F.when(F.col("capacity_int") >= 200, F.lit(0.05)).otherwise(F.lit(0.0))
    )).cast(FloatType())
)

# COMMAND ----------
# MAGIC %md ## 24. Healthcare Capability Dependency Graph

# COMMAND ----------

CAPABILITY_GRAPH_VERSION = "capability_graph_v1"

CAPABILITY_DEPENDENCY_RULES: Dict[str, Dict[str, Any]] = {
    "icu": {
        "requires": {
            "oxygen": ["oxygen", "ventilator", "respirator"],
            "patient_monitoring": ["monitor", "icu", "critical care"],
            "trained_staff": ["intensivist", "critical care", "anaesth", "nurse"],
            "beds": ["bed", "ward", "capacity"],
            "emergency_support": ["emergency", "trauma", "casualty"],
        }
    },
    "surgery": {
        "requires": {
            "operating_theatre": ["theatre", "surgery", "surgical", "operating"],
            "anesthesia": ["anesthesia", "anaesthesia", "anaesth"],
            "sterilization": ["steril", "autoclave"],
            "blood_support": ["blood bank", "transfusion", "laboratory"],
            "recovery_beds": ["recovery", "ward", "bed", "capacity"],
        }
    },
    "emergency": {
        "requires": {
            "triage": ["triage", "casualty", "emergency"],
            "ambulance_or_referral": ["ambulance", "referral", "transport"],
            "oxygen": ["oxygen", "resuscitation"],
            "trauma_stabilization": ["trauma", "stabilization", "resuscitation"],
            "diagnostics": ["x-ray", "xray", "ultrasound", "laboratory", "lab"],
        }
    },
    "obstetrics": {
        "requires": {
            "delivery_room": ["delivery", "labour", "labor ward", "maternity"],
            "midwifery": ["midwife", "midwifery", "obstetric"],
            "neonatal_support": ["neonatal", "nicu", "incubator", "newborn"],
            "blood_support": ["blood bank", "transfusion", "laboratory"],
        }
    },
    "radiology": {
        "requires": {
            "imaging_equipment": ["x-ray", "xray", "ct", "mri", "ultrasound", "radiology"],
            "trained_staff": ["radiologist", "radiographer", "imaging"],
        }
    },
    "infectious_disease": {
        "requires": {
            "laboratory": ["laboratory", "lab", "pcr", "hiv", "tb", "malaria"],
            "infection_prevention": ["isolation", "ipc", "infection prevention"],
            "pharmacy_supply": ["pharmacy", "antiretroviral", "art", "vaccine"],
        }
    },
    "mental_health": {
        "requires": {
            "trained_staff": ["psychiatrist", "psychologist", "mental health", "counsel"],
            "counseling_space": ["counsel", "therapy", "psychiatric"],
            "referral_chain": ["referral", "community", "outreach"],
        }
    },
}

GRAPH_SCHEMA = StructType([
    StructField("capability_graph_nodes", StringType(), True),
    StructField("capability_graph_edges", StringType(), True),
    StructField("capability_dependency_gaps", StringType(), True),
    StructField("capability_graph_summary", StringType(), True),
    StructField("service_richness_score", FloatType(), True),
    StructField("infrastructure_completeness_score", FloatType(), True),
    StructField("referral_complexity_score", FloatType(), True),
    StructField("healthcare_maturity_score", FloatType(), True),
])


def _graph_list(value: Any) -> List[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip()]
    if isinstance(value, np.ndarray):
        return [str(v).strip() for v in value.tolist() if str(v).strip()]
    text = str(value).strip()
    if not text or text.lower() in ("null", "none", "nan", "[]"):
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return [str(v).strip() for v in parsed if str(v).strip()]
    except Exception:
        pass
    return [text]


def _has_any(text: str, needles: List[str]) -> bool:
    low = text.lower()
    return any(n in low for n in needles)


@F.udf(GRAPH_SCHEMA)
def capability_graph_udf(
    specialties, procedures, equipment, capabilities, description,
    has_icu, has_surgery, has_emergency_medicine, has_obstetrics,
    has_radiology, has_infectious_disease, has_mental_health,
    number_doctors_int, capacity_int, facility_type_clean, facility_tier_label,
    emergency_readiness_score, critical_care_score,
):
    specs = _graph_list(specialties)
    procs = _graph_list(procedures)
    equip = _graph_list(equipment)
    caps = _graph_list(capabilities)
    evidence_text = " ".join(specs + procs + equip + caps + [str(description or "")]).lower()

    active_domains: Dict[str, bool] = {
        "icu": bool(has_icu) or _has_any(evidence_text, ["icu", "intensive care", "critical care"]),
        "surgery": bool(has_surgery) or _has_any(evidence_text, ["surgery", "surgical", "theatre", "operation"]),
        "emergency": bool(has_emergency_medicine) or _has_any(evidence_text, ["emergency", "trauma", "casualty"]),
        "obstetrics": bool(has_obstetrics) or _has_any(evidence_text, ["maternity", "obstetric", "delivery", "labour"]),
        "radiology": bool(has_radiology) or _has_any(evidence_text, ["radiology", "x-ray", "xray", "ultrasound", "imaging"]),
        "infectious_disease": bool(has_infectious_disease) or _has_any(evidence_text, ["hiv", "tb", "malaria", "infectious"]),
        "mental_health": bool(has_mental_health) or _has_any(evidence_text, ["psychiatric", "mental health", "counsel"]),
    }

    doctors = int(number_doctors_int or 0)
    beds = int(capacity_int or 0)
    nodes = set()
    edges: List[Dict[str, str]] = []
    gaps: List[str] = []
    required_count = 0
    met_count = 0

    facility_label = str(facility_tier_label or facility_type_clean or "facility")
    nodes.add(f"facility:{facility_label}")
    for spec in specs:
        nodes.add(f"specialty:{spec}")
        edges.append({"from": "facility", "to": f"specialty:{spec}", "relation": "offers", "status": "observed"})

    for domain, is_active in active_domains.items():
        if not is_active:
            continue
        nodes.add(f"service:{domain}")
        edges.append({"from": "facility", "to": f"service:{domain}", "relation": "claims_or_infers", "status": "active"})
        for dep, keywords in CAPABILITY_DEPENDENCY_RULES[domain]["requires"].items():
            required_count += 1
            dependency_met = _has_any(evidence_text, keywords)
            if dep == "trained_staff" and doctors > 0:
                dependency_met = True
            if dep in ("beds", "recovery_beds") and beds > 0:
                dependency_met = True
            if dependency_met:
                met_count += 1
            else:
                gaps.append(f"{domain}:{dep}")
            nodes.add(f"dependency:{dep}")
            edges.append({
                "from": f"service:{domain}",
                "to": f"dependency:{dep}",
                "relation": "requires",
                "status": "met" if dependency_met else "missing_evidence",
            })

    service_richness = min(1.0, (len(specs) + len(procs) + len(equip) + len(caps)) / 24.0)
    infra_completeness = float(met_count / required_count) if required_count else 0.0
    referral_complexity = min(1.0, (sum(active_domains.values()) / 7.0) + (0.15 if "referral" in facility_label.lower() else 0.0))
    maturity = min(1.0, (
        0.30 * service_richness +
        0.30 * infra_completeness +
        0.20 * float(emergency_readiness_score or 0.0) +
        0.20 * float(critical_care_score or 0.0)
    ))
    summary = {
        "active_domains": [k for k, v in active_domains.items() if v],
        "required_dependencies": required_count,
        "met_dependencies": met_count,
        "missing_dependencies": len(gaps),
        "graph_version": CAPABILITY_GRAPH_VERSION,
    }
    return (
        json.dumps(sorted(nodes)),
        json.dumps(edges),
        json.dumps(gaps),
        json.dumps(summary),
        round(service_richness, 4),
        round(infra_completeness, 4),
        round(referral_complexity, 4),
        round(maturity, 4),
    )


gold = gold.withColumn(
    "_capability_graph",
    capability_graph_udf(
        F.col("specialties_parsed"), F.col("procedure_parsed"),
        F.col("equipment_parsed"), F.col("capability_parsed"),
        F.col("description"),
        F.col("has_icu"), F.col("has_surgery"), F.col("has_emergency_medicine"),
        F.col("has_obstetrics"), F.col("has_radiology"),
        F.col("has_infectious_disease"), F.col("has_mental_health"),
        F.col("number_doctors_int"), F.col("capacity_int"),
        F.col("facility_type_clean"), F.col("facility_tier_label"),
        F.col("emergency_readiness_score"), F.col("critical_care_score"),
    )
)

for _field in GRAPH_SCHEMA.fieldNames():
    gold = gold.withColumn(_field, F.col(f"_capability_graph.{_field}"))
gold = gold.drop("_capability_graph").withColumn("capability_graph_version", F.lit(CAPABILITY_GRAPH_VERSION))

# COMMAND ----------
# MAGIC %md ## 25. Healthcare Desert Score (v11-F6: base credit 0.20)

# COMMAND ----------

gold = gold.withColumn(
    "_spec_access",
    (
        F.col("has_emergency_medicine") .cast(FloatType()) * 0.18
        + F.col("has_surgery")          .cast(FloatType()) * 0.14
        + F.col("has_obstetrics")       .cast(FloatType()) * 0.12
        + F.col("has_icu")              .cast(FloatType()) * 0.10
        + F.col("has_radiology")        .cast(FloatType()) * 0.07
        + F.col("has_pediatrics")       .cast(FloatType()) * 0.06
        + F.col("has_infectious_disease").cast(FloatType()) * 0.05
        + F.col("has_mental_health")    .cast(FloatType()) * 0.04
    )
)

gold = gold.withColumn(
    "_base_access",
    (
        # v11-F6: base credit raised to 0.20 for fully operational + clinical data present
        F.when(
            F.col("is_operational") & (
                F.col("has_capabilities") | F.col("has_procedures") | F.col("has_specialties")
            ),
            F.lit(0.20),
        ).when(
            F.col("has_capabilities") | F.col("has_procedures") | F.col("has_specialties"),
            F.lit(0.07)
        ).otherwise(F.lit(0.0))
        + F.least(F.lit(0.06), F.col("capability_count").cast(FloatType()) / 10.0 * 0.06)
        + F.least(F.lit(0.04), F.col("procedure_count").cast(FloatType()) / 6.0 * 0.04)
        + F.least(F.lit(0.08), F.coalesce(F.col("capacity_int").cast(FloatType()) / 1500.0, F.lit(0.0)))
        + F.when(
            F.col("number_doctors_int").isNotNull() & (F.col("number_doctors_int") > 0),
            F.lit(0.04)
          ).otherwise(F.lit(0.0))
    )
)

gold = gold.withColumn(
    "_geo_factor",
    F.when(F.col("geo_precision_tier") >= 3, F.lit(1.00))   # v11: city dict = tier 3 → full credit
     .when(F.col("geo_precision_tier") == 2, F.lit(0.75))
     .when(F.col("geo_precision_tier") == 1, F.lit(0.45))
     .otherwise(F.lit(0.40)).cast(FloatType())
)

gold = gold.withColumn(
    "_op_factor",
    F.when(F.col("operational_status") == "open",    F.lit(1.00))
     .when(F.col("operational_status") == "unknown", F.lit(0.82))
     .when(F.col("operational_status") == "closed",  F.lit(0.10))
     .otherwise(F.lit(0.82)).cast(FloatType())
)

gold = gold.withColumn(
    "medical_desert_score",
    (
        F.lit(1.0) - F.least(
            F.lit(1.0),
            (F.col("_spec_access") + F.col("_base_access"))
            * F.col("_geo_factor")
            * F.col("_op_factor")
            * F.when(F.col("evidence_weight") < 0.50, F.lit(0.90)).otherwise(F.lit(1.0))
        )
    ).cast(FloatType())
)

gold = gold.withColumn(
    "desert_label",
    # G3: Recalibrated — actual data mean=0.711, so spread labels proportionally
    # Old: ≥0.92 Severe, ≥0.78 Moderate, ≥0.60 Marginal, ≥0.40 Adequate, else Well-Served
    # New: ≥0.88 Severe, ≥0.72 Moderate, ≥0.55 Marginal, ≥0.35 Adequate, else Well-Served
    F.when(F.col("medical_desert_score") >= 0.88, F.lit("Severe Desert"))
     .when(F.col("medical_desert_score") >= 0.72, F.lit("Moderate Desert"))
     .when(F.col("medical_desert_score") >= 0.55, F.lit("Marginal"))
     .when(F.col("medical_desert_score") >= 0.35, F.lit("Adequate"))
     .otherwise(F.lit("Well-Served"))
)

gold = gold.drop("_spec_access", "_base_access", "_geo_factor", "_op_factor")

print("Desert label distribution (v11-F6: base credit 0.20):")
gold.groupBy("desert_label").count().orderBy(F.desc("count")).show()
gold.agg(
    F.round(F.avg("medical_desert_score"), 3).alias("avg"),
    F.round(F.min("medical_desert_score"), 3).alias("min"),
    F.round(F.max("medical_desert_score"), 3).alias("max"),
).show()

# COMMAND ----------
# MAGIC %md ## 25. RAG Quality Score + Tiered RAG Readiness

# COMMAND ----------

gold = gold.withColumn(
    "rag_quality_score",
    F.greatest(
        F.lit(0.0),
        F.least(F.lit(1.0), (
            F.when(F.col("doc_text_length") >= 500, F.lit(0.25))
             .when(F.col("doc_text_length") >= 200, F.lit(0.15))
             .when(F.col("doc_text_length") >= 80,  F.lit(0.08))
             .otherwise(F.lit(0.0))
            + F.when(F.col("procedure_count") >= 3, F.lit(0.10)).otherwise(F.lit(0.0))
            + F.when(F.col("equipment_count") >= 2, F.lit(0.08)).otherwise(F.lit(0.0))
            + F.when(F.col("capability_count") >= 2, F.lit(0.07)).otherwise(F.lit(0.0))
            + F.when(F.col("specialty_count") >= 2, F.lit(0.05)).otherwise(F.lit(0.0))
            + F.when(F.col("source_trust") == "structured",    F.lit(0.20))
               .when(F.col("source_trust") == "text_extracted",F.lit(0.14))
               .when(F.col("source_trust") == "inferred",      F.lit(0.08))
               .otherwise(F.lit(0.04))
            + F.when(F.col("geo_precision_tier") >= 4, F.lit(0.15))
               .when(F.col("geo_precision_tier") == 3, F.lit(0.12))  # v11: city dict → good score
               .when(F.col("geo_precision_tier") == 2, F.lit(0.05))
               .otherwise(F.lit(0.0))
            + F.when(F.col("has_description"), F.lit(0.10)).otherwise(F.lit(0.0))
            - F.when(
                F.arrays_overlap(
                    F.col("row_quality_flags"),
                    F.array(F.lit("MEDIUM:capability_contamination_risk"))
                ),
                F.lit(0.10)
              ).otherwise(F.lit(0.0))
        ))
    ).cast(FloatType())
)

gold = gold.withColumn(
    "is_rag_ready",
    (F.col("rag_quality_score") >= 0.20)
    & (F.col("doc_text_length") > 80)
    & (
        F.col("has_procedures")
        | F.col("has_equipment")
        | F.col("has_specialties")
        | (F.col("has_capabilities") & (F.col("capability_count") >= 2))
    )
    & (F.col("geo_source") != "country_centroid")
    & ~F.arrays_overlap(
        F.col("row_quality_flags"),
        F.array(F.lit("HIGH:no_clinical_data"), F.lit("HIGH:missing_name"))
    )
)

gold = gold \
    .withColumn("is_search_ready",   F.col("is_rag_ready") & F.col("has_contact")) \
    .withColumn("is_planning_ready", F.col("is_rag_ready") & (F.col("geo_precision_tier") >= 3)) \
    .withColumn("is_clinical_ready", F.col("is_rag_ready") & F.col("has_procedures") & F.col("has_specialties"))

_rag_ct   = gold.filter(F.col("is_rag_ready")).count()
_srch_ct  = gold.filter(F.col("is_search_ready")).count()
_plan_ct  = gold.filter(F.col("is_planning_ready")).count()
_clin_ct  = gold.filter(F.col("is_clinical_ready")).count()
print(f"RAG-ready:        {_rag_ct:,} / {gold.count():,}  ({_rag_ct/gold.count()*100:.1f}%)")
print(f"  search_ready:   {_srch_ct:,}  |  planning_ready: {_plan_ct:,}  |  clinical_ready: {_clin_ct:,}")

# COMMAND ----------
# MAGIC %md ## 26. Ghost Probability Score (v11-F7)

# COMMAND ----------

gold = gold.withColumn(
    "ghost_probability_score",
    F.least(
        F.lit(1.0),
        (
            F.when(F.col("data_completeness_score") < 0.35, F.lit(0.28)).otherwise(F.lit(0.0))
            + F.when(~F.col("has_contact"),                   F.lit(0.18)).otherwise(F.lit(0.0))
            + F.when(F.col("procedure_count") == 0,           F.lit(0.13)).otherwise(F.lit(0.0))
            + F.when(F.col("equipment_count") == 0,           F.lit(0.08)).otherwise(F.lit(0.0))
            # G4: social_metadata penalty cut in half when procedure_count > 0
            + F.when(
                (F.col("source_trust") == "social_metadata") & (F.col("procedure_count") == 0) & (F.col("equipment_count") == 0),
                F.lit(0.15)
              ).when(
                (F.col("source_trust") == "social_metadata"),
                F.lit(0.06)  # G4: was 0.08 flat; halved when clinical evidence exists
              ).otherwise(F.lit(0.0))
            + F.when(F.col("geo_source") == "country_centroid",  F.lit(0.18)).otherwise(F.lit(0.0))
            + F.when(F.col("geo_source") == "region_centroid",   F.lit(0.05)).otherwise(F.lit(0.0))
            + F.when(~F.col("has_physical_address"),              F.lit(0.04)).otherwise(F.lit(0.0))
            + F.when(~F.col("has_description"),                   F.lit(0.04)).otherwise(F.lit(0.0))
        )
    ).cast(FloatType())
)

gold = gold.withColumn(
    "ghost_review_priority",
    F.when(F.col("ghost_probability_score") >= 0.55, F.lit("HIGH"))
     .when(F.col("ghost_probability_score") >= 0.38, F.lit("MEDIUM"))
     .when(F.col("ghost_probability_score") >= 0.22, F.lit("LOW"))
     .otherwise(F.lit("NONE"))
)

# Validate ghost mean
_ghost_mean = gold.agg(F.avg("ghost_probability_score")).collect()[0][0] or 0.0
print(f"Ghost probability mean: {_ghost_mean:.3f}  (target < 0.30 for realistic dataset)")
print("Ghost review priority distribution (v12 recalibrated):")
gold.groupBy("ghost_review_priority").count().orderBy(F.desc("count")).show()

# COMMAND ----------
# MAGIC %md ## 27. Citation Row ID + Gold Citation Upgrade

# COMMAND ----------

gold = gold.withColumn(
    "citation_row_id",
    F.md5(F.concat_ws(
        "|",
        F.coalesce(F.col("unique_id"),   F.lit("")),
        F.coalesce(F.col("name"),        F.lit("")),
        F.coalesce(F.col("ingested_at").cast(StringType()), F.lit("")),
    ))
)

GOLD_CITATION_SCHEMA = ArrayType(StructType([
    StructField("field",             StringType(), True),
    StructField("snippet",           StringType(), True),
    StructField("source_column",     StringType(), True),
    StructField("extraction_method", StringType(), True),
    StructField("confidence",        FloatType(),  True),
    StructField("step_id",           StringType(), True),
]))

_SRC_STEP_MAP = {
    "procedure":          "gold_v11_procedure_enrichment",
    "equipment":          "gold_v11_equipment_enrichment",
    "capability":         "gold_v11_capability_validation",
    "specialties":        "gold_v11_specialty_inference",
    "description":        "gold_v11_description_fallback",
    "procedure_parsed":   "gold_v11_procedure_enrichment",
    "equipment_parsed":   "gold_v11_equipment_enrichment",
    "capability_parsed":  "gold_v11_capability_validation",
    "specialties_parsed": "gold_v11_specialty_inference",
}

_FLAG_SOURCES = [
    ("has_emergency_medicine", "emergencyMedicine",       "specialties"),
    ("has_obstetrics",         "gynecologyAndObstetrics", "specialties"),
    ("has_surgery",            "generalSurgery",          "specialties"),
    ("has_pediatrics",         "pediatrics",              "specialties"),
    ("has_icu",                "criticalCareMedicine",    "specialties"),
    ("has_radiology",          "radiology",               "specialties"),
    ("has_infectious_disease", "infectiousDiseases",      "specialties"),
    ("has_mental_health",      "psychiatry",              "specialties"),
]


def _upgrade_citations(
    silver_citations, proc_arr, equip_arr, cap_arr, spec_arr,
    has_em, has_obs, has_surg, has_ped,
    has_icu_v, has_rad, has_inf, has_mh, evidence_wt,
):
    out, seen = [], set()
    for c in (silver_citations or []):
        if not c:
            continue
        try:
            field  = getattr(c, "field",             None) or ""
            snip   = getattr(c, "snippet",           None) or ""
            src    = getattr(c, "source_column",     None) or ""
            method = getattr(c, "extraction_method", None) or ""
            conf   = float(getattr(c, "confidence",  None) or 0.0)
        except:
            continue
        conf = min(1.0, conf * float(evidence_wt or 1.0))
        step = _SRC_STEP_MAP.get(src, f"gold_v11_{src}")
        key  = (field, snip[:50])
        if key not in seen:
            seen.add(key)
            out.append({
                "field": field, "snippet": snip[:300], "source_column": src,
                "extraction_method": method, "confidence": float(conf), "step_id": step,
            })

    fired_flags = [has_em, has_obs, has_surg, has_ped, has_icu_v, has_rad, has_inf, has_mh]
    for (flag_name, snip_val, src_col), fired in zip(_FLAG_SOURCES, fired_flags):
        if not fired:
            continue
        snippet = snip_val
        for arr in (spec_arr, cap_arr, proc_arr, equip_arr):
            if arr:
                for item in arr:
                    if item and snip_val.lower() in str(item).lower():
                        snippet = str(item)[:100]
                        break
        key = (flag_name, snippet[:50])
        if key not in seen:
            seen.add(key)
            conf = min(0.92, 0.92 * float(evidence_wt or 1.0))
            out.append({
                "field": flag_name, "snippet": snippet[:300], "source_column": src_col,
                "extraction_method": "transitive_specialty_flag", "confidence": float(conf),
                "step_id": f"gold_v11_flag_{flag_name}",
            })

    if not out:
        out.append({
            "field": "description", "snippet": "", "source_column": "description",
            "extraction_method": "no_evidence", "confidence": 0.0,
            "step_id": "gold_v11_no_evidence",
        })
    return out


upgrade_citations_udf = F.udf(_upgrade_citations, GOLD_CITATION_SCHEMA)
gold = gold.withColumn(
    "citations",
    upgrade_citations_udf(
        F.col("citations"), F.col("procedure_parsed"), F.col("equipment_parsed"),
        F.col("capability_parsed"), F.col("specialties_parsed"),
        F.col("has_emergency_medicine"), F.col("has_obstetrics"),
        F.col("has_surgery"),           F.col("has_pediatrics"),
        F.col("has_icu"),               F.col("has_radiology"),
        F.col("has_infectious_disease"), F.col("has_mental_health"),
        F.col("evidence_weight"),
    ),
)

# COMMAND ----------
# MAGIC %md ## 28. Update extraction_version

# COMMAND ----------

gold = gold.withColumn("extraction_version", F.lit(cfg.EXTRACTION_VER))

# COMMAND ----------
# MAGIC %md ## 29. Final Column Selection — Gold Schema (179 columns)
# MAGIC
# MAGIC v11+ adds schema-safe facility graph fields for dependency-aware RAG/anomaly reasoning.

# COMMAND ----------

GOLD_COLUMNS = [
    # ── 1. Identity / Region (6) ─────────────────────────────────────────
    "region_normalised",
    "unique_id", "source_url", "name", "pk_unique_id", "mongo_db",
    # ── 2. Raw clinical JSON strings (6) ─────────────────────────────────
    "specialties", "procedure", "equipment", "capability",
    "organization_type", "content_table_id",
    # ── 3. Contact — phone_numbers is ARRAY<STRING> in gold (11) ─────────
    "phone_numbers",
    "email", "websites", "officialWebsite",
    "yearestablished", "acceptsvolunteers",
    "facebooklink", "twitterlink", "linkedinlink", "instagramlink", "logo",
    # ── 4. Address (9) ───────────────────────────────────────────────────
    "address_line1", "address_line2", "address_line3",
    "address_city", "address_stateOrRegion", "address_zipOrPostcode",
    "address_country", "address_countryCode", "countries",
    # ── 5. Organisation text (3) ─────────────────────────────────────────
    "missionstatement", "missionstatementlink", "organizationdescription",
    # ── 6. Type IDs (3) ──────────────────────────────────────────────────
    "facilityTypeId", "operatorTypeId", "affiliationtypeids",
    # ── 7. Clinical text + numerics (4) ──────────────────────────────────
    "description", "area", "numberDoctors", "capacity",
    # ── 8. Provenance (5) ────────────────────────────────────────────────
    "ingested_at", "source_file", "dataset_version", "country_scope", "row_hash",
    # ── 9. Parsed arrays (8) ─────────────────────────────────────────────
    "specialties_parsed", "procedure_parsed", "equipment_parsed", "capability_parsed",
    "phone_numbers_parsed", "affiliationtypeids_parsed",
    "countries_parsed", "websites_parsed",
    # ── 10. Derived single-value fields (6) ──────────────────────────────
    "official_phone", "area_int",
    "year_established_int", "year_confidence",
    "accepts_volunteers_bool", "pk_unique_id_int",
    # ── 11. Facility / operator type classification (7) ──────────────────
    "facility_type_raw", "operator_type_raw",
    "facility_type_clean", "facility_type_clean_pdf", "facility_type_confidence",
    "organization_type_clean", "organization_category",
    # ── 12. Ownership booleans (3) ───────────────────────────────────────
    "is_ngo", "is_faith_based", "is_government",
    # ── 13. Location (12) ────────────────────────────────────────────────
    "city_clean", "region_source", "region_confidence",
    "latitude", "longitude", "geo_source", "geo_confidence",
    "postal_address", "has_physical_address", "address_type",
    # ── 14. Clinical numerics (4) ────────────────────────────────────────
    "capacity_int", "capacity_confidence",
    "number_doctors_int", "doctors_confidence",
    # ── 15. Operational (2) ──────────────────────────────────────────────
    "operational_status", "is_operational",
    # ── 16. Trust & dedup (7) ────────────────────────────────────────────
    "source_trust", "has_bare_website_domain",
    "dedup_key", "dedup_cluster_size", "is_duplicate_survivor",
    "is_generated_canonical", "canonical_source_group",
    # ── 17. Content flags + counts (12) ──────────────────────────────────
    "has_procedures", "has_equipment", "has_capabilities",
    "has_specialties", "has_description", "has_contact",
    "procedure_count", "equipment_count", "capability_count", "specialty_count",
    "phone_count", "has_multiple_phones",
    # ── 18. RAG (3) ──────────────────────────────────────────────────────
    "doc_text", "doc_text_length", "is_rag_ready",
    # ── 19. Citations + quality (4) ──────────────────────────────────────
    "citations",
    "row_quality_flags", "quality_flag_count", "data_completeness_score",
    # ── 20. Versioning (1) ───────────────────────────────────────────────
    "extraction_version",
    # ════════════════════════════════════════════════════════════════════════
    # ══ GOLD-ONLY ADDITIONS (54) ════════════════════════════════════════════
    # ════════════════════════════════════════════════════════════════════════
    # ── 21. Org category provenance (3) — v11 adds organization_category_inferred
    "organization_category_confidence", "organization_category_source",
    "organization_category_inferred",
    # ── 22. Ownership model (1) ──────────────────────────────────────────
    "ownership_model",
    # ── 23. Specialty presence flags (8) ─────────────────────────────────
    "has_emergency_medicine", "has_obstetrics", "has_surgery", "has_pediatrics",
    "has_icu", "has_radiology", "has_infectious_disease", "has_mental_health",
    # ── 24. Specialty tracking (2) ───────────────────────────────────────
    "specialty_direct_count", "specialty_inferred_count",
    # ── 25. Facility classification booleans (4) ─────────────────────────
    "is_public", "is_private", "is_hospital", "is_clinic",
    # ── 26. Hospital sub-classification (4) ──────────────────────────────
    "is_teaching_hospital", "is_referral_center",
    "is_military_hospital", "is_specialist_hospital",
    # ── 27. Facility sophistication (3) — v11 adds facility_tier_label ───
    "facility_complexity_level", "clinical_complexity_score",
    "facility_tier_label",
    # ── 28. Geo quality (4) — v11 adds geo_region_mismatch ───────────────
    "geo_precision_tier", "geo_quality_score",
    "geo_contradiction_flag", "geo_region_mismatch",
    # ── 29. Evidence weighting (1) ───────────────────────────────────────
    "evidence_weight",
    # ── 30. Statistical anomaly flags (7) ────────────────────────────────
    "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
    "stat_anomaly_clinic_claims_icu",    "stat_anomaly_ghost_facility",
    "stat_anomaly_specialty_mismatch",   "stat_anomaly_procedure_breadth",
    "total_stat_anomalies",
    # ── 31. Ghost scoring (2) ────────────────────────────────────────────
    "ghost_probability_score", "ghost_review_priority",
    # ── 32. Quality risk scores (4) ──────────────────────────────────────
    "quality_risk_score", "clinical_risk_score",
    "operational_risk_score", "integrity_risk_score",
    # ── 33. Dimensioned completeness (3) ─────────────────────────────────
    "clinical_completeness", "location_completeness", "contact_completeness",
    # ── 34. NGO + citation ID (2) ────────────────────────────────────────
    "ngo_serves_ghana", "citation_row_id",
    # ── 35. RAG quality + tiered readiness (4) ───────────────────────────
    "rag_quality_score", "is_search_ready", "is_planning_ready", "is_clinical_ready",
    # ── 36. Desert score (2) ─────────────────────────────────────────────
    "medical_desert_score", "desert_label",
    # ── 37. Emergency readiness + critical care (2) ──────────────────────
    "emergency_readiness_score", "critical_care_score",
    # ── 38. Capability graph and maturity signals (9) ───────────────────
    "capability_graph_nodes", "capability_graph_edges",
    "capability_dependency_gaps", "capability_graph_summary",
    "service_richness_score", "infrastructure_completeness_score",
    "referral_complexity_score", "healthcare_maturity_score",
    "capability_graph_version",
]

# ── Validation ────────────────────────────────────────────────────────────
_missing_gc = [c for c in GOLD_COLUMNS if c not in gold.columns]
if _missing_gc:
    raise ValueError(f"MISSING columns before write: {_missing_gc}")

_seen_gc, _dup_gc = set(), []
for _c in GOLD_COLUMNS:
    if _c in _seen_gc:
        _dup_gc.append(_c)
    _seen_gc.add(_c)
assert not _dup_gc, f"Duplicate columns in GOLD_COLUMNS: {_dup_gc}"

gold_final = gold.select(*GOLD_COLUMNS)
_actual_cnt   = len(gold_final.columns)
_expected_cnt = len(GOLD_COLUMNS)
print(f"Column count : {_actual_cnt}  (expected {_expected_cnt})")
assert _actual_cnt == _expected_cnt, f"Column count mismatch: {_actual_cnt} vs {_expected_cnt}"
print("Column schema validated ✓")

# COMMAND ----------
# MAGIC %md ## 30. Write gold_facilities_enriched

# COMMAND ----------

(
    gold_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact",   "true")
    .saveAsTable(cfg.GOLD_FACILITIES)
)

_readback = spark.table(cfg.GOLD_FACILITIES)
_rc = _readback.count()
_cc = len(_readback.columns)
print(f"✅  {cfg.GOLD_FACILITIES}")
print(f"    Rows    : {_rc:,}  (survivors only; {_audit_count} duplicates in audit)")
print(f"    Columns : {_cc}")
assert _rc == gold_final.count()
assert _cc == _expected_cnt

from pyspark.sql.types import NullType
_void = [f.name for f in _readback.schema.fields if isinstance(f.dataType, NullType)]
if _void:
    print(f"⚠️  VOID columns: {_void}")
else:
    print("✅  No VOID columns detected.")

spark.sql(f"""
    COMMENT ON TABLE {cfg.GOLD_FACILITIES}
    IS 'Gold v11: BUG FIXED transitive enrichment, numpy import, geo upgrade null-safety,
        teaching→ICU hard rule, facility_tier_label, geo_region_mismatch,
        org_category_inferred, improved contact completeness — {cfg.EXTRACTION_VER}'
""")

# COMMAND ----------
# MAGIC %md ## 31. Build gold_regional_summary (70 columns)

# COMMAND ----------

gf = spark.table(cfg.GOLD_FACILITIES)

CRITICAL_SPECS_5 = [
    "emergencyMedicine", "generalSurgery",
    "gynecologyAndObstetrics", "pediatrics", "internalMedicine",
]

_POP_MAP = F.create_map(*[F.lit(x) for pair in GHANA_POPULATION.items() for x in pair])

_CANONICAL_SPEC_SET = frozenset([
    "internalMedicine", "familyMedicine", "pediatrics", "cardiology",
    "generalSurgery", "emergencyMedicine", "gynecologyAndObstetrics",
    "orthopedicSurgery", "dentistry", "ophthalmology", "radiology",
    "pathology", "infectiousDiseases", "nephrology", "criticalCareMedicine",
    "cardiacSurgery", "plasticSurgery", "neurology", "psychiatry", "anesthesia",
    "dermatology", "urology", "gastroenterology", "pulmonology",
    "endocrinologyAndDiabetesAndMetabolism", "neonatologyPerinatalMedicine",
    "medicalOncology", "physicalMedicineAndRehabilitation", "otolaryngology",
    "geriatricsInternalMedicine", "hospiceAndPalliativeInternalMedicine",
    "publicHealth", "globalHealthAndInternationalHealth",
    "maternalFetalMedicineOrPerinatology", "socialAndBehavioralSciences",
    "obstetricsAndMaternityCare", "reproductiveEndocrinologyAndInfertility",
    "clinicalPathology", "diagnosticRadiology", "interventionalRadiology",
    "neurosurgery", "pediatricSurgery", "vascularSurgery",
])

regional = gf.groupBy("region_normalised").agg(
    # Facility counts
    F.count("unique_id")                                                          .alias("total_facilities"),
    F.sum(F.when(F.col("is_hospital") | F.col("is_clinic"), 1).otherwise(0))     .alias("clinical_facility_count"),
    F.sum(F.col("is_hospital").cast(IntegerType()))                               .alias("hospital_count"),
    F.sum(F.when(
        F.col("is_hospital") & (F.col("has_procedures") | F.col("has_equipment") | F.col("has_capabilities")),
        1).otherwise(0))                                                          .alias("clinical_hospital_count"),
    F.sum(F.col("is_clinic").cast(IntegerType()))                                 .alias("clinic_count"),
    F.sum(F.col("is_public").cast(IntegerType()))                                 .alias("public_facilities"),
    F.sum(F.col("is_private").cast(IntegerType()))                                .alias("private_facilities"),
    F.sum(F.col("is_ngo").cast(IntegerType()))                                    .alias("ngo_count"),
    F.sum(F.col("is_faith_based").cast(IntegerType()))                            .alias("faith_based_count"),
    F.sum(F.col("is_teaching_hospital").cast(IntegerType()))                      .alias("teaching_hospital_count"),
    F.sum(F.col("is_referral_center").cast(IntegerType()))                        .alias("referral_center_count"),
    F.sum(F.col("is_specialist_hospital").cast(IntegerType()))                    .alias("specialist_hospital_count"),  # v11
    # Medical resources
    F.avg("number_doctors_int")                                                   .alias("avg_doctors"),
    F.sum(F.coalesce("number_doctors_int", F.lit(0)))                             .alias("total_doctors"),
    F.avg("capacity_int")                                                         .alias("avg_bed_capacity"),
    F.sum(F.coalesce("capacity_int", F.lit(0)))                                   .alias("total_beds"),
    # Quality metrics
    F.avg("data_completeness_score")                                              .alias("avg_completeness"),
    F.avg("geo_quality_score")                                                    .alias("avg_geo_quality"),
    F.avg("clinical_complexity_score")                                            .alias("avg_clinical_complexity"),
    F.avg("evidence_weight")                                                      .alias("avg_evidence_weight"),
    F.avg("ghost_probability_score")                                              .alias("avg_ghost_probability"),  # v11
    # Specialty facility counts
    F.sum(F.col("has_emergency_medicine").cast(IntegerType()))                    .alias("emergency_medicine_facilities"),
    F.sum(F.col("has_obstetrics")        .cast(IntegerType()))                    .alias("obstetrics_facilities"),
    F.sum(F.col("has_surgery")           .cast(IntegerType()))                    .alias("surgery_facilities"),
    F.sum(F.col("has_pediatrics")        .cast(IntegerType()))                    .alias("pediatrics_facilities"),
    F.sum(F.col("has_icu")               .cast(IntegerType()))                    .alias("icu_facilities"),
    F.sum(F.col("has_infectious_disease").cast(IntegerType()))                    .alias("infectious_disease_facilities"),
    F.sum(F.col("has_radiology")         .cast(IntegerType()))                    .alias("radiology_facilities"),
    F.sum(F.col("has_mental_health")     .cast(IntegerType()))                    .alias("mental_health_facilities"),
    # Infrastructure coverage
    F.sum(F.col("has_procedures") .cast(IntegerType()))                           .alias("facilities_with_procedures"),
    F.sum(F.col("has_equipment")  .cast(IntegerType()))                           .alias("facilities_with_equipment"),
    F.sum(F.col("has_capabilities").cast(IntegerType()))                          .alias("facilities_with_capabilities"),
    F.sum(F.when(F.col("accepts_volunteers_bool") == True, 1).otherwise(0))      .alias("volunteer_facilities"),
    # Geospatial
    F.first(F.col("latitude"),  ignorenulls=True)                                 .alias("region_centroid_lat"),
    F.first(F.col("longitude"), ignorenulls=True)                                 .alias("region_centroid_lon"),
    # Anomaly + quality
    F.sum("total_stat_anomalies")                                                 .alias("total_region_anomalies"),
    F.avg("quality_risk_score")                                                   .alias("avg_quality_risk"),
    F.avg("medical_desert_score")                                                 .alias("avg_facility_desert_score"),
    F.avg("emergency_readiness_score")                                            .alias("avg_emergency_readiness"),
    F.avg("critical_care_score")                                                  .alias("avg_critical_care_score"),
    F.avg("service_richness_score")                                               .alias("avg_service_richness_score"),
    F.avg("infrastructure_completeness_score")                                    .alias("avg_infrastructure_completeness_score"),
    F.avg("referral_complexity_score")                                            .alias("avg_referral_complexity_score"),
    F.avg("healthcare_maturity_score")                                            .alias("avg_healthcare_maturity_score"),
    # RAG readiness
    F.sum(F.col("is_rag_ready").cast(IntegerType()))                              .alias("rag_ready_count"),
    F.sum(F.col("is_clinical_ready").cast(IntegerType()))                         .alias("clinical_ready_count"),
    # Specialty aggregation
    F.collect_list("specialties_parsed")                                          .alias("_all_spec_arrays"),
    # Ownership breakdown
    F.sum(F.when(F.col("ownership_model") == "government", 1).otherwise(0))      .alias("government_facilities"),
    F.sum(F.when(F.col("ownership_model") == "faith_based", 1).otherwise(0))     .alias("regional_faith_based_count"),
)


@F.udf(ArrayType(StringType()))
def flatten_filter_specs(all_arrays):
    if not all_arrays:
        return []
    seen, out = set(), []
    for sub_arr in (all_arrays or []):
        if not sub_arr:
            continue
        for s in sub_arr:
            if s and s in _CANONICAL_SPEC_SET and s not in seen:
                seen.add(s)
                out.append(s)
    return sorted(out)


regional = regional \
    .withColumn("all_specialties", flatten_filter_specs(F.col("_all_spec_arrays"))) \
    .drop("_all_spec_arrays")

regional = regional.withColumn(
    "rag_ready_rate",
    F.round(F.col("rag_ready_count").cast(FloatType()) /
            F.greatest(F.lit(1), F.col("total_facilities")) * 100.0, 1).cast(FloatType())
)


@F.udf(ArrayType(StringType()))
def missing_critical_udf(all_specs):
    present = set(all_specs or [])
    return [s for s in CRITICAL_SPECS_5 if s not in present]


regional = regional \
    .withColumn("missing_critical_specialties", missing_critical_udf(F.col("all_specialties"))) \
    .withColumn("critical_specialty_gap_count",
                F.size(F.col("missing_critical_specialties")).cast(IntegerType()))

regional = regional.withColumn(
    "region_population",
    F.coalesce(_POP_MAP[F.col("region_normalised")], F.lit(1_000_000)).cast(LongType())
)

regional = (
    regional
    .withColumn("facilities_per_100k",
        F.round(F.col("total_facilities").cast(FloatType()) / F.col("region_population") * 100000, 2))
    .withColumn("hospitals_per_100k",
        F.round(F.col("hospital_count").cast(FloatType()) / F.col("region_population") * 100000, 2))
    .withColumn("beds_per_100k",
        F.round(F.col("total_beds").cast(FloatType()) / F.col("region_population") * 100000, 2))
    .withColumn("doctors_per_100k",
        F.round(F.col("total_doctors").cast(FloatType()) / F.col("region_population") * 100000, 2))
    .withColumn("icu_facilities_per_100k",
        F.round(F.col("icu_facilities").cast(FloatType()) / F.col("region_population") * 100000, 3))
    .withColumn("surgery_facilities_per_100k",
        F.round(F.col("surgery_facilities").cast(FloatType()) / F.col("region_population") * 100000, 3))
    .withColumn("maternity_facilities_per_100k",
        F.round(F.col("obstetrics_facilities").cast(FloatType()) / F.col("region_population") * 100000, 3))
    .withColumn("public_private_ratio",
        F.round(F.col("public_facilities").cast(FloatType()) /
                F.greatest(F.lit(1), F.col("private_facilities")), 3))
)

regional = (
    regional
    .withColumn("maternity_gap_score",
        F.greatest(F.lit(0.0),
            F.lit(1.0) - F.col("obstetrics_facilities").cast(FloatType()) /
            F.greatest(F.lit(1), F.col("clinical_facility_count"))
        ).cast(FloatType()))
    .withColumn("emergency_gap_score",
        F.greatest(F.lit(0.0),
            F.lit(1.0) - F.col("emergency_medicine_facilities").cast(FloatType()) /
            F.greatest(F.lit(1), F.col("hospital_count"))
        ).cast(FloatType()))
    .withColumn("icu_gap_score",
        F.when(F.col("icu_facilities") == 0, F.lit(1.0))
         .when(F.col("icu_facilities") < 2,  F.lit(0.70))
         .when(F.col("icu_facilities") < 5,  F.lit(0.40))
         .otherwise(F.lit(0.20)).cast(FloatType()))
    .withColumn("surgical_access_gap_score",
        F.greatest(F.lit(0.0),
            F.lit(1.0) - F.col("surgery_facilities").cast(FloatType()) /
            F.greatest(F.lit(1), F.col("hospital_count"))
        ).cast(FloatType()))
    .withColumn("public_private_imbalance_score",
        F.greatest(F.lit(0.0),
            F.lit(1.0) - F.col("public_facilities").cast(FloatType()) /
            F.greatest(F.lit(1), F.col("total_facilities"))
        ).cast(FloatType()))
)

regional = regional.withColumn(
    "medical_desert_score",
    (
        F.lit(1.0) - F.least(F.lit(1.0), (
            F.col("emergency_medicine_facilities").cast(FloatType()) /
            F.greatest(F.lit(1), F.col("hospital_count")) * 0.25
            + F.col("surgery_facilities").cast(FloatType()) /
              F.greatest(F.lit(1), F.col("hospital_count")) * 0.20
            + F.col("obstetrics_facilities").cast(FloatType()) /
              F.greatest(F.lit(1), F.col("clinical_facility_count")) * 0.18
            + F.when(F.col("icu_facilities") > 0, F.lit(0.15)).otherwise(F.lit(0.0))
            + F.when(F.col("beds_per_100k") >= 10, F.lit(0.10)).otherwise(
                F.col("beds_per_100k").cast(FloatType()) / 10.0 * 0.10)
            + F.when(F.col("hospitals_per_100k") >= 2, F.lit(0.12)).otherwise(
                F.col("hospitals_per_100k").cast(FloatType()) / 2.0 * 0.12)
        ))
    ).cast(FloatType())
)

regional = regional.withColumn(
    "desert_label",
    F.when(F.col("medical_desert_score") >= 0.85, F.lit("Severe Desert"))
     .when(F.col("medical_desert_score") >= 0.68, F.lit("Moderate Desert"))
     .when(F.col("medical_desert_score") >= 0.48, F.lit("Marginal"))
     .when(F.col("medical_desert_score") >= 0.28, F.lit("Adequate"))
     .otherwise(F.lit("Well-Served"))
)


@F.udf(ArrayType(StringType()))
def recommended_actions_udf(
    missing_specs, avg_doctors, total_beds, icu_fac,
    hospital_count, mental_fac, total_fac, hospitals_per_100k, beds_per_100k,
):
    actions = []
    missing = list(missing_specs or [])
    if "emergencyMedicine" in missing:
        actions.append("URGENT: Deploy emergency medicine capacity — zero coverage detected")
    if "generalSurgery" in missing and (hospital_count or 0) > 0:
        actions.append("URGENT: No surgical capacity — patients cannot receive operative care")
    if "gynecologyAndObstetrics" in missing:
        actions.append("URGENT: No obstetrics — elevated maternal mortality risk")
    if "pediatrics" in missing:
        actions.append("HIGH: Deploy paediatric care — children's health services absent")
    if "internalMedicine" in missing:
        actions.append("HIGH: Recruit internal medicine specialists")
    if (avg_doctors or 0.0) < 0.5:
        actions.append("URGENT: Critical physician shortage — avg < 1 doctor per facility")
    if (total_beds or 0) < 20:
        actions.append("HIGH: Deploy mobile inpatient capacity — fewer than 20 beds in region")
    if (icu_fac or 0) == 0 and (hospital_count or 0) > 0:
        actions.append("HIGH: No ICU capacity — critical gap for complex cases")
    if (mental_fac or 0) == 0:
        actions.append("MEDIUM: Mental health services absent — underserved population")
    if (hospitals_per_100k or 0.0) < 0.5:
        actions.append("HIGH: Hospital density critically low — < 0.5 per 100k population")
    if (beds_per_100k or 0.0) < 2.0:
        actions.append("HIGH: Bed capacity insufficient — fewer than 2 beds per 100k")
    if (total_fac or 0) < 5:
        actions.append("URGENT: Region severely underserved — prioritise facility establishment")
    return actions or ["Monitor — no critical access gaps identified at this time"]


regional = regional.withColumn(
    "recommended_actions",
    recommended_actions_udf(
        F.col("missing_critical_specialties"),
        F.col("avg_doctors"), F.col("total_beds"),
        F.col("icu_facilities"), F.col("hospital_count"),
        F.col("mental_health_facilities"), F.col("total_facilities"),
        F.col("hospitals_per_100k"), F.col("beds_per_100k"),
    ),
)

# ── 66-column regional schema (64 v10 + 2 new: specialist_hospital_count, avg_ghost_probability)
REGIONAL_COLUMNS = [
    "region_normalised",
    "total_facilities", "clinical_facility_count", "hospital_count",
    "clinical_hospital_count", "clinic_count",
    "public_facilities", "private_facilities", "ngo_count",
    "faith_based_count", "government_facilities", "regional_faith_based_count",
    "teaching_hospital_count", "referral_center_count",
    "specialist_hospital_count",           # v11 new
    "avg_doctors", "total_doctors", "avg_bed_capacity", "total_beds",
    "avg_completeness", "avg_geo_quality", "avg_clinical_complexity", "avg_evidence_weight",
    "avg_ghost_probability",               # v11 new
    "emergency_medicine_facilities", "obstetrics_facilities",
    "surgery_facilities", "pediatrics_facilities", "icu_facilities",
    "infectious_disease_facilities", "radiology_facilities", "mental_health_facilities",
    "facilities_with_procedures", "facilities_with_equipment",
    "facilities_with_capabilities", "volunteer_facilities",
    "region_centroid_lat", "region_centroid_lon",
    "total_region_anomalies", "avg_quality_risk", "avg_facility_desert_score",
    "avg_emergency_readiness", "avg_critical_care_score",
    "avg_service_richness_score", "avg_infrastructure_completeness_score",
    "avg_referral_complexity_score", "avg_healthcare_maturity_score",
    "rag_ready_count", "rag_ready_rate", "clinical_ready_count",
    "region_population",
    "facilities_per_100k", "hospitals_per_100k", "beds_per_100k", "doctors_per_100k",
    "icu_facilities_per_100k", "surgery_facilities_per_100k", "maternity_facilities_per_100k",
    "public_private_ratio",
    "maternity_gap_score", "emergency_gap_score", "icu_gap_score",
    "surgical_access_gap_score", "public_private_imbalance_score",
    "all_specialties", "missing_critical_specialties",
    "critical_specialty_gap_count", "recommended_actions",
    "medical_desert_score", "desert_label",
]

_miss_reg = [c for c in REGIONAL_COLUMNS if c not in regional.columns]
if _miss_reg:
    raise ValueError(f"Missing regional columns: {_miss_reg}")

regional_final = regional.select(*REGIONAL_COLUMNS)
_reg_col_cnt   = len(regional_final.columns)
print(f"Regional columns: {_reg_col_cnt}  (expected {len(REGIONAL_COLUMNS)})")
assert _reg_col_cnt == len(REGIONAL_COLUMNS)
assert len(set(REGIONAL_COLUMNS)) == len(REGIONAL_COLUMNS), "Duplicate regional cols"
print("Regional schema validated ✓")

# COMMAND ----------
# MAGIC %md ## 32. Write gold_regional_summary

# COMMAND ----------

(
    regional_final.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(cfg.GOLD_REGIONAL)
)

_reg_cnt = spark.table(cfg.GOLD_REGIONAL).count()
print(f"✅  {cfg.GOLD_REGIONAL}")
print(f"    Rows    : {_reg_cnt}  (expected 16–18 regions)")
print(f"    Columns : {_reg_col_cnt}")

spark.sql(f"""
    COMMENT ON TABLE {cfg.GOLD_REGIONAL}
    IS 'Regional summary v11: 66 cols, specialist_hospital_count, avg_ghost_probability — {cfg.EXTRACTION_VER}'
""")

# COMMAND ----------
# MAGIC %md ## 33. Gold Quality Report

# COMMAND ----------

gf  = spark.table(cfg.GOLD_FACILITIES)
gr  = spark.table(cfg.GOLD_REGIONAL)
tot = gf.count()

print(f"{'='*74}")
print(f"  GOLD FACILITIES v11 — Quality Report")
print(f"  Table   : {cfg.GOLD_FACILITIES}")
print(f"  Rows    : {tot:,}  (survivor-only; {_audit_count} duplicates in audit table)")
print(f"  Columns : {len(gf.columns)}")
print(f"{'='*74}")

_checks = [
    ("is_hospital",                              gf.filter(F.col("is_hospital")).count()),
    ("is_clinic",                                gf.filter(F.col("is_clinic")).count()),
    ("is_ngo",                                   gf.filter(F.col("is_ngo") == True).count()),
    ("is_faith_based",                           gf.filter(F.col("is_faith_based") == True).count()),
    ("is_government",                            gf.filter(F.col("is_government") == True).count()),
    ("is_teaching_hospital",                     gf.filter(F.col("is_teaching_hospital")).count()),
    ("is_referral_center",                       gf.filter(F.col("is_referral_center")).count()),
    ("is_military_hospital",                     gf.filter(F.col("is_military_hospital")).count()),
    ("is_specialist_hospital",                   gf.filter(F.col("is_specialist_hospital")).count()),
    ("has_emergency_medicine",                   gf.filter(F.col("has_emergency_medicine")).count()),
    ("has_obstetrics",                           gf.filter(F.col("has_obstetrics")).count()),
    ("has_surgery",                              gf.filter(F.col("has_surgery")).count()),
    ("has_icu [v11 fixed — was 16 in v10]",      gf.filter(F.col("has_icu")).count()),
    ("has_radiology",                            gf.filter(F.col("has_radiology")).count()),
    ("has_infectious_disease",                   gf.filter(F.col("has_infectious_disease")).count()),
    ("has_mental_health",                        gf.filter(F.col("has_mental_health")).count()),
    ("is_rag_ready",                             gf.filter(F.col("is_rag_ready")).count()),
    ("is_search_ready",                          gf.filter(F.col("is_search_ready")).count()),
    ("is_planning_ready",                        gf.filter(F.col("is_planning_ready")).count()),
    ("is_clinical_ready",                        gf.filter(F.col("is_clinical_ready")).count()),
    ("has_physical_address (v11 enhanced)",      gf.filter(F.col("has_physical_address")).count()),
    ("geo_region_mismatch [new v11]",            gf.filter(F.col("geo_region_mismatch")).count()),
    ("geo_source = extended_city_dict",          gf.filter(F.col("geo_source") == "extended_city_dict").count()),
    ("geo_source = region_centroid",             gf.filter(F.col("geo_source") == "region_centroid").count()),
    ("geo_source = country_centroid",            gf.filter(F.col("geo_source") == "country_centroid").count()),
    ("org_category = NULL (target = 0)",         gf.filter(F.col("organization_category").isNull()).count()),
    ("org_category_inferred [new v11]",          gf.filter(F.col("organization_category_inferred")).count()),
    ("org_category confidence = high",           gf.filter(F.col("organization_category_confidence") == "high").count()),
    ("stat_anomaly_hospital_no_doctors (v11)",   gf.filter(F.col("stat_anomaly_hospital_no_doctors")).count()),
    ("stat_anomaly_ghost_facility",              gf.filter(F.col("stat_anomaly_ghost_facility")).count()),
    ("ghost_review_priority = HIGH",             gf.filter(F.col("ghost_review_priority") == "HIGH").count()),
    ("ghost_review_priority = MEDIUM",           gf.filter(F.col("ghost_review_priority") == "MEDIUM").count()),
    ("total_stat_anomalies > 0",                 gf.filter(F.col("total_stat_anomalies") > 0).count()),
]

for _label, _cnt in _checks:
    _pct = _cnt / tot * 100 if tot else 0
    print(f"  {_label:<55}  {_cnt:>5,}  ({_pct:5.1f}%)")

print()
print("Facility tier label distribution (v11 new):")
gf.groupBy("facility_tier_label").count().orderBy(F.desc("count")).show()

print("Facility complexity distribution:")
gf.groupBy("facility_complexity_level").count().orderBy("facility_complexity_level").show()

print("Desert label distribution (v11: base credit 0.20):")
gf.groupBy("desert_label").count().orderBy(F.desc("count")).show()

print("Ownership model distribution:")
gf.groupBy("ownership_model").count().orderBy(F.desc("count")).show()

print("Organization category distribution:")
gf.groupBy("organization_category").count().orderBy(F.desc("count")).show()

print("Score statistics (v11):")
gf.agg(
    F.round(F.avg("clinical_complexity_score"),   3).alias("avg_clinical_complexity"),
    F.round(F.avg("geo_quality_score"),           3).alias("avg_geo_quality"),
    F.round(F.avg("rag_quality_score"),           3).alias("avg_rag_quality"),
    F.round(F.avg("quality_risk_score"),          3).alias("avg_quality_risk"),
    F.round(F.avg("operational_risk_score"),      3).alias("avg_operational_risk"),
    F.round(F.avg("medical_desert_score"),        3).alias("avg_desert_score"),
    F.round(F.avg("evidence_weight"),             3).alias("avg_evidence_weight"),
    F.round(F.avg("ghost_probability_score"),     3).alias("avg_ghost_prob"),
    F.round(F.avg("emergency_readiness_score"),   3).alias("avg_emergency_readiness"),
    F.round(F.avg("critical_care_score"),         3).alias("avg_critical_care"),
).show()

print(f"{'='*74}")
print(f"  GOLD REGIONAL SUMMARY v11  ({gr.count()} regions, {len(gr.columns)} columns)")
print(f"{'='*74}")

display(
    gr.select(
        "region_normalised",
        "total_facilities", "hospital_count",
        "hospitals_per_100k", "beds_per_100k",
        "emergency_medicine_facilities", "icu_facilities",
        "critical_specialty_gap_count",
        "maternity_gap_score", "emergency_gap_score",
        "rag_ready_rate", "public_private_ratio",
        "avg_ghost_probability",
        "medical_desert_score", "desert_label",
    ).orderBy(F.desc("medical_desert_score"))
)

# COMMAND ----------
# MAGIC %md ## 34. Row-Level Diagnostics

# COMMAND ----------

gf = spark.table(cfg.GOLD_FACILITIES)

print("── v11 FIX VERIFICATION: Teaching/referral hospitals with ICU ─────────")
gf.filter(F.col("is_teaching_hospital") | F.col("is_referral_center")) \
  .select("name", "region_normalised", "is_teaching_hospital", "is_referral_center",
          "has_icu", "has_surgery", "has_emergency_medicine",
          "facility_complexity_level", "clinical_complexity_score") \
  .orderBy(F.desc("clinical_complexity_score")).show(20, truncate=60)

print("── Top 15 HIGH-RISK ghost facilities ─────────────────────────────────")
gf.filter(F.col("ghost_review_priority").isin("HIGH", "MEDIUM")) \
  .select("name", "region_normalised", "facility_type_clean",
          "ghost_probability_score", "ghost_review_priority",
          "data_completeness_score", "geo_source", "has_contact",
          "capability_count", "procedure_count") \
  .orderBy(F.desc("ghost_probability_score")).show(15, truncate=55)

print("── Geo region mismatches (v11 new) ──────────────────────────────────")
gf.filter(F.col("geo_region_mismatch")) \
  .select("name", "city_clean", "region_normalised", "geo_source", "geo_quality_score") \
  .orderBy("region_normalised").show(20, truncate=55)

print("── Multi-anomaly rows ────────────────────────────────────────────────")
gf.filter(F.col("total_stat_anomalies") >= 2) \
  .select("name", "region_normalised",
          "stat_anomaly_capability_inflation", "stat_anomaly_hospital_no_doctors",
          "stat_anomaly_procedure_breadth", "total_stat_anomalies",
          "data_completeness_score") \
  .orderBy(F.desc("total_stat_anomalies"), F.asc("data_completeness_score")) \
  .show(15, truncate=55)

print("── Country-centroid geocodes (action needed) ─────────────────────────")
gf.filter(F.col("geo_source") == "country_centroid") \
  .select("name", "region_normalised", "geo_quality_score",
          "has_physical_address", "address_line1", "city_clean") \
  .orderBy(F.desc("data_completeness_score")) \
  .show(15, truncate=55)

print("── Org category breakdown with inference source ──────────────────────")
gf.groupBy("organization_category", "organization_category_source",
           "organization_category_confidence") \
  .count().orderBy(F.desc("count")).show(20)

print("── Audit table top clusters ──────────────────────────────────────────")
_audit_df = spark.table(cfg.GOLD_AUDIT)
print(f"Audit table rows: {_audit_df.count()}")
_audit_df.groupBy("name").count().orderBy(F.desc("count")).show(10, truncate=60)

# COMMAND ----------
# MAGIC %md ## 35. MLflow Tracing

# COMMAND ----------

try:
    import mlflow

    mlflow.set_experiment("/Users/dasdeepayan08@gmail.com/virtue-foundation-idp-hackathon")

    with mlflow.start_run(run_name="03_gold_build_v11") as run:
        gf  = spark.table(cfg.GOLD_FACILITIES)
        gr  = spark.table(cfg.GOLD_REGIONAL)
        tot = gf.count()

        mlflow.log_metric("total_facilities",        tot)
        mlflow.log_metric("audit_duplicates",        _audit_count)
        mlflow.log_metric("survivor_rate",           round(tot / _raw_count * 100, 2))
        mlflow.log_metric("hospital_count",          gf.filter(F.col("is_hospital")).count())
        mlflow.log_metric("teaching_hospital_count", gf.filter(F.col("is_teaching_hospital")).count())
        mlflow.log_metric("rag_ready_count",         gf.filter(F.col("is_rag_ready")).count())
        mlflow.log_metric("search_ready_count",      gf.filter(F.col("is_search_ready")).count())
        mlflow.log_metric("clinical_ready_count",    gf.filter(F.col("is_clinical_ready")).count())
        mlflow.log_metric("unknown_region_pct",      round(_unk_ct / _survivor_count * 100, 2))
        mlflow.log_metric("org_cat_null_count",
                          gf.filter(F.col("organization_category").isNull()).count())
        mlflow.log_metric("has_icu_count",           gf.filter(F.col("has_icu")).count())
        mlflow.log_metric("geo_region_mismatch_count",
                          gf.filter(F.col("geo_region_mismatch")).count())
        mlflow.log_metric("country_centroid_rows",   gf.filter(F.col("geo_source") == "country_centroid").count())
        mlflow.log_metric("extended_city_dict_rows", gf.filter(F.col("geo_source") == "extended_city_dict").count())
        mlflow.log_metric("region_centroid_rows",    gf.filter(F.col("geo_source") == "region_centroid").count())
        mlflow.log_metric("physical_address_count",  gf.filter(F.col("has_physical_address")).count())
        mlflow.log_metric("severe_desert_count",     gf.filter(F.col("desert_label") == "Severe Desert").count())
        mlflow.log_metric("moderate_desert_count",   gf.filter(F.col("desert_label") == "Moderate Desert").count())
        mlflow.log_metric("hospital_no_doctors_count",
                          gf.filter(F.col("stat_anomaly_hospital_no_doctors")).count())

        for _f in ["has_emergency_medicine", "has_obstetrics", "has_surgery", "has_pediatrics",
                   "has_icu", "has_radiology", "has_infectious_disease", "has_mental_health"]:
            mlflow.log_metric(_f, gf.filter(F.col(_f)).count())

        _avgs = gf.agg(
            F.avg("clinical_complexity_score"), F.avg("geo_quality_score"),
            F.avg("rag_quality_score"),          F.avg("medical_desert_score"),
            F.avg("evidence_weight"),            F.avg("emergency_readiness_score"),
            F.avg("critical_care_score"),
        ).collect()[0]
        mlflow.log_metric("avg_clinical_complexity",   round(float(_avgs[0] or 0), 4))
        mlflow.log_metric("avg_geo_quality",           round(float(_avgs[1] or 0), 4))
        mlflow.log_metric("avg_rag_quality",           round(float(_avgs[2] or 0), 4))
        mlflow.log_metric("avg_desert_score",          round(float(_avgs[3] or 0), 4))
        mlflow.log_metric("avg_evidence_weight",       round(float(_avgs[4] or 0), 4))
        mlflow.log_metric("avg_emergency_readiness",   round(float(_avgs[5] or 0), 4))
        mlflow.log_metric("avg_critical_care_score",   round(float(_avgs[6] or 0), 4))

        mlflow.log_param("gold_facilities_table", cfg.GOLD_FACILITIES)
        mlflow.log_param("gold_audit_table",      cfg.GOLD_AUDIT)
        mlflow.log_param("gold_regional_table",   cfg.GOLD_REGIONAL)
        mlflow.log_param("extraction_version",    cfg.EXTRACTION_VER)
        mlflow.log_param("silver_input_rows",     str(_raw_count))
        mlflow.log_param("survivors_written",     str(tot))
        mlflow.log_param("gold_col_count",        str(len(gf.columns)))
        mlflow.log_param("regional_col_count",    str(len(gr.columns)))
        mlflow.log_param("v11_bug_fixes",         "transitive_udf_applied,numpy_import,geo_null_safety,icu_hard_rule")

        print(f"✅  MLflow run logged: {run.info.run_id}")

except Exception as e:
    print(f"MLflow skipped (non-fatal): {e}")


In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_facilities_enriched 

In [0]:
%sql
SELECT * FROM virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_regional_summary

In [0]:
%sql
SHOW CREATE TABLE virtue_foundation.ghana.gold_facilities_enriched